# TD Prediction Pipeline - Replication Notebook

**Purpose.** Reproduces Stages 1-10 of the experiments described in the accompanying MSc thesis on High-Risk Technical Debt prediction. Given a copy of the Technical Debt Dataset v2.0 (`td_V2.db`) on Google Drive, running this notebook top-to-bottom regenerates every experimental artefact: cleaned parquet tables, label / feature / dataset matrices, within-project and cross-project results, sensitivity and ablation tables, SHAP-based feature-importance, all seven thesis figures, and the auto-generated `docs/06_results.md` / `docs/07_discussion.md`.

**Author.** Abdulmajid Awol Seid  
**Version.** 1.0 (2026-04-26)  
**Scope.** Experimentation only. Hand-written thesis prose chapters are *not* produced or copied by this notebook.


## Pipeline overview

```mermaid
flowchart TB
    Front[Front matter and reproducibility setup] --> Cfg[Config cell] --> Mods[src/ modules writefile]
    Mods --> S1[Stage 1: DB inspection]
    S1 --> S2[Stage 2: Snapshot selection]
    S2 --> S3[Stage 3: Clean and basename normalise]
    S3 --> S4[Stage 4: Three-variant labelling and agreement]
    S4 --> S5[Stage 5: Static and historical features]
    S5 --> S6[Stage 6: Dataset assembly and leakage audit]
    S6 --> S7[Stage 7: Within-project 10-fold CV]
    S7 --> S8[Stage 8: LOPO cross-project CV]
    S8 --> S9[Stage 9: Sensitivity grid and ablation]
    S9 --> S10[Stage 10: SHAP, figures, results.md, discussion.md]
    S10 --> Final[Runtime receipt and zip output to Drive]
```

### Runtime expectations (default Colab CPU runtime)

| Stage | Description                                  | Wall-clock | Peak RAM |
|-------|----------------------------------------------|------------|----------|
| 1     | DB schema inventory                          | ~10 s      | < 1 GB   |
| 2     | Per-project snapshot selection               | ~30 s      | < 1 GB   |
| 3     | Clean and persist as parquet                 | ~3 min     | ~3 GB    |
| 4     | Three-variant labelling                      | ~45 s      | ~2 GB    |
| 5     | Static + historical feature extraction       | ~30 s      | ~2 GB    |
| 6     | Dataset assembly + leakage audit             | ~5 s       | ~1 GB    |
| 7     | Within-project 10-fold CV (5 models x 3)     | ~2 min     | ~2 GB    |
| 8     | LOPO cross-project CV (5 models x 3)         | ~3 min     | ~2 GB    |
| 9     | Sensitivity grid + ablation                  | ~3 min     | ~2 GB    |
| 10    | SHAP + figures + render markdown reports     | ~3 min     | ~3 GB    |
| **Total** |                                          | **~15 min** |          |


## Data citation

Cite the underlying dataset (Lenarduzzi, Saarimaki and Taibi 2019) when reusing any output of this notebook:

```bibtex
@inproceedings{Lenarduzzi2019TDDataset,
  author    = {Lenarduzzi, Valentina and Saarim{\"a}ki, Nyyti and Taibi, Davide},
  title     = {The Technical Debt Dataset},
  booktitle = {Proceedings of the 15th International Conference on Predictive Models and Data Analytics in Software Engineering (PROMISE)},
  year      = {2019},
  doi       = {10.1145/3345629.3345630},
  publisher = {ACM}
}
```

Dataset URL: <https://github.com/clowee/The-Technical-Debt-Dataset/releases>

## Table of contents

1. [Reproducibility setup](#repro-setup)
2. [Configuration](#configuration)
3. [Source modules](#source-modules)
4. [Stage 1 - Database inspection](#stage-1)
5. [Stage 2 - Snapshot selection](#stage-2)
6. [Stage 3 - Clean and basename-normalise](#stage-3)
7. [Stage 4 - Three-variant labelling](#stage-4)
8. [Stage 5 - Feature extraction](#stage-5)
9. [Stage 6 - Dataset assembly](#stage-6)
10. [Stage 7 - Within-project 10-fold CV](#stage-7)
11. [Stage 8 - LOPO cross-project CV](#stage-8)
12. [Stage 9 - Sensitivity and ablation](#stage-9)
13. [Stage 10 - SHAP, figures and reports](#stage-10)
14. [Finalisation - runtime, zip, copy to Drive](#finalisation)


<a id="repro-setup"></a>
## Reproducibility setup

Five cells: (A) detect Colab and mount Drive; (B) verify the SQLite database; (C) install pinned dependencies; (D) set deterministic seeds and capture an environment receipt; (E) scaffold `/content/` and symlink the database into `data/raw/`.

All cells are idempotent - re-running them does not duplicate work.

In [ ]:
# Cell A: detect Colab and mount Drive (idempotent)
import sys, os
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/td_pipeline')
else:
    # Local fallback so the notebook is testable outside Colab
    DRIVE_ROOT = Path.cwd() / 'td_pipeline_drive'
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('IN_COLAB =', IN_COLAB)
print('DRIVE_ROOT =', DRIVE_ROOT)


In [ ]:
# Cell B: assert the SQLite database is on Drive and record an integrity
# fingerprint (SHA-256 of the first 64 MiB - full hash on a 1.5 GB file is
# unnecessarily expensive; the prefix is a sufficient sanity check).
import hashlib

DB_PATH = DRIVE_ROOT / 'td_V2.db'
assert DB_PATH.exists(), (
    f'Place td_V2.db in {DRIVE_ROOT} before running this notebook.\n'
    'Download from https://github.com/clowee/The-Technical-Debt-Dataset/releases'
)
size_mb = DB_PATH.stat().st_size / (1024 ** 2)
print(f'td_V2.db size: {size_mb:.1f} MB')

h = hashlib.sha256()
with open(DB_PATH, 'rb') as f:
    bytes_hashed = 0
    while bytes_hashed < 64 * 1024 * 1024:
        chunk = f.read(1024 * 1024)
        if not chunk:
            break
        h.update(chunk)
        bytes_hashed += len(chunk)
DB_FINGERPRINT = h.hexdigest()[:16]
print(f'SHA256 prefix (first {bytes_hashed // (1024*1024)} MiB): {DB_FINGERPRINT}')


In [ ]:
# Cell C: install pinned dependencies. Skipped silently when each package
# is already importable, so re-runs are nearly instant.
import importlib.util

REQUIRED = [
    ('pandas', 'pandas>=2.0.0'),
    ('numpy', 'numpy>=1.24.0'),
    ('sklearn', 'scikit-learn>=1.3.0'),
    ('xgboost', 'xgboost>=2.0.0'),
    ('lightgbm', 'lightgbm>=4.0.0'),
    ('imblearn', 'imbalanced-learn>=0.11.0'),
    ('matplotlib', 'matplotlib>=3.7.0'),
    ('seaborn', 'seaborn>=0.12.0'),
    ('pyarrow', 'pyarrow>=14.0.0'),
    ('shap', 'shap>=0.44.0'),
    ('matplotlib_venn', 'matplotlib-venn>=0.11.9'),
    ('tabulate', 'tabulate>=0.9.0'),
    ('scipy', 'scipy>=1.11.0'),
    ('tqdm', 'tqdm>=4.65.0'),
]
missing = [pin for mod, pin in REQUIRED if importlib.util.find_spec(mod) is None]
if missing:
    print('Installing missing packages:', missing)
    import subprocess
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', *missing],
        check=True,
    )
else:
    print('All dependencies already satisfied; skipping pip install.')


In [ ]:
# Cell D: deterministic seeds + environment receipt for the appendix.
import json, platform, random, datetime
import numpy as np
import importlib.metadata as md_meta

RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
os.environ['PYTHONHASHSEED'] = str(RANDOM_STATE)

_pkgs = ['pandas', 'numpy', 'scikit-learn', 'xgboost', 'lightgbm', 'shap',
         'matplotlib', 'seaborn', 'pyarrow', 'tabulate', 'matplotlib-venn',
         'imbalanced-learn', 'tqdm', 'scipy']
_versions = {}
for p in _pkgs:
    try:
        _versions[p] = md_meta.version(p)
    except md_meta.PackageNotFoundError:
        _versions[p] = None

ENV_RECEIPT = {
    'timestamp_utc': datetime.datetime.utcnow().isoformat() + 'Z',
    'python_version': sys.version.split()[0],
    'platform': platform.platform(),
    'in_colab': IN_COLAB,
    'random_state': RANDOM_STATE,
    'db_fingerprint_sha256_prefix': DB_FINGERPRINT,
    'db_size_mb': round(size_mb, 1),
    'package_versions': _versions,
}
print(json.dumps(ENV_RECEIPT, indent=2))


In [ ]:
# Cell E: scaffold the project tree under /content and symlink the
# database into /content/data/raw so config.py finds it.
import shutil

CONTENT = Path('/content') if IN_COLAB else (Path.cwd() / 'colab_workdir')
CONTENT.mkdir(parents=True, exist_ok=True)

for sub in [
    'src/data', 'src/features', 'src/models', 'src/analysis', 'src/reporting',
    'data/raw', 'data/processed', 'data/external',
    'results/tables', 'results/figures',
    'docs',
]:
    (CONTENT / sub).mkdir(parents=True, exist_ok=True)

# Empty package markers so `from src.data import ...` works
for pkg_dir in ['src', 'src/data', 'src/features', 'src/models', 'src/analysis', 'src/reporting']:
    init = CONTENT / pkg_dir / '__init__.py'
    if not init.exists():
        init.write_text('', encoding='utf-8')

DB_LINK = CONTENT / 'data' / 'raw' / 'td_V2.db'
if not DB_LINK.exists():
    try:
        DB_LINK.symlink_to(DB_PATH)
        print(f'Symlinked DB: {DB_LINK} -> {DB_PATH}')
    except (OSError, NotImplementedError) as exc:
        print(f'Symlink unavailable ({exc}); copying instead (slow on first run).')
        shutil.copy2(DB_PATH, DB_LINK)
else:
    print(f'DB already linked at {DB_LINK}')

if str(CONTENT) not in sys.path:
    sys.path.insert(0, str(CONTENT))

# Persist the env receipt now that results/ exists
(CONTENT / 'results' / 'env_receipt.json').write_text(
    json.dumps(ENV_RECEIPT, indent=2), encoding='utf-8'
)

# Wall-clock accumulator used by every stage
RUNTIMES: dict[str, float] = {}
RECOMPUTE = False  # set True to force re-execution of cached stages

print('Working tree at:', CONTENT)


<a id="configuration"></a>
## Configuration

The next cell writes `config.py` to `/content/`. It is the single source of truth for paths, snapshot policy, labelling parameters, feature catalogues and model definitions.

Reviewers wishing to vary the experiment without editing source files should change one of the parameters below in the **config cell itself** and re-run from Stage 4 onwards (Stages 1-3 do not depend on these):

- `OBSERVATION_WINDOW_MONTHS` (primary 6) - length of the post-snapshot window for label derivation.
- `HIGH_RISK_PERCENTILE` (primary 20) - top P% by risk score flagged as positive.
- `MIN_PRE_SNAPSHOT_COMMITS` (500) and `MIN_POST_SNAPSHOT_COMMITS` (50) - eligibility thresholds.
- `SENSITIVITY_WINDOWS` and `SENSITIVITY_PERCENTILES` - grids used by Stage 9.
- `RISK_SCORE_WEIGHTS` - weighting between bug-fix commits, future churn and SZZ events in the consequence risk score.
- `MODELS` and `RANDOM_STATE` - controlled training settings.

After editing, save and continue running the notebook; the change propagates because every downstream cell imports from this `config.py`.


In [ ]:
%%writefile /content/config.py
"""
Configuration for the Technical Debt Prediction Research pipeline.

Aligned with the approved MSc research proposal (Updated, Feb 2026) by
Abdulmajid Awol Seid. This module centralizes paths, temporal-split
parameters, three-variant labeling settings, feature catalogues, and
machine-learning model specifications.

Sections
--------
1. Paths
2. Dataset
3. Temporal split and snapshot policy
4. Labeling (three variants + sensitivity grid)
5. Feature catalogues (Proposal Table 1)
6. Modeling (models, hyperparameter grids, imbalance handling)
7. Evaluation (metrics, CV folds, randomness)
"""
from __future__ import annotations

from pathlib import Path

# ---------------------------------------------------------------------------
# 1. Paths
# ---------------------------------------------------------------------------
PROJECT_ROOT = Path(__file__).parent
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
TABLES_DIR = RESULTS_DIR / "tables"

DOCS_DIR = PROJECT_ROOT / "docs"
SCRIPTS_DIR = PROJECT_ROOT / "scripts"

# Ensure the output subtrees exist at import time (idempotent).
for _d in (PROCESSED_DATA_DIR, EXTERNAL_DATA_DIR, FIGURES_DIR, TABLES_DIR, DOCS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# 2. Dataset
# ---------------------------------------------------------------------------
# Technical Debt Dataset v2.0 (Lenarduzzi et al. 2019).
# Download: https://github.com/clowee/The-Technical-Debt-Dataset/releases
TD_DATASET_PATH = RAW_DATA_DIR / "td_V2.db"

# Java source files only; exclude tests, generated code, build artefacts.
SOURCE_FILE_EXTENSIONS = (".java",)
PATH_EXCLUSION_PATTERNS = (
    "/test/",
    "/tests/",
    "/generated/",
    "/generated-sources/",
    "/target/",
    "/build/",
)

# ---------------------------------------------------------------------------
# 3. Temporal split and snapshot policy
# ---------------------------------------------------------------------------
# Snapshot selection strategy per project. "median" is the primary choice:
# the median commit date of the project's master-branch history. Alternative
# strategies can be activated by overriding this in a script if needed.
SNAPSHOT_STRATEGY = "median"  # one of: "median", "release", "fixed"

# Primary observation window (months after snapshot) used to derive labels.
OBSERVATION_WINDOW_MONTHS = 6

# Sensitivity analysis windows (months) to assess robustness of labeling.
SENSITIVITY_WINDOWS = (3, 6, 12)

# Minimum history requirements for a project to be usable.
# `min_post=50` chosen to maximize LOPO fold count while preserving label
# signal quality (expected top-20% positives per project: 20-50 files).
# Rationale documented in RESEARCH_LOG.md, 2026-04-23 entry.
MIN_PRE_SNAPSHOT_COMMITS = 500
MIN_POST_SNAPSHOT_COMMITS = 50

# ---------------------------------------------------------------------------
# 4. Labeling (three variants + sensitivity)
# ---------------------------------------------------------------------------
# Primary labeling: consequence-oriented. High-Risk TD = top P% by maintenance
# risk score computed over the post-snapshot observation window.
HIGH_RISK_PERCENTILE = 20  # top 20%

# Sensitivity analysis thresholds.
SENSITIVITY_PERCENTILES = (10, 20, 30)

# Weights for the consequence risk score components. See Proposal Section 3.4.
# All components are min-max normalized within project before weighting.
RISK_SCORE_WEIGHTS = {
    "bugfix_commits_future": 0.5,
    "future_churn": 0.3,
    "szz_defects_future": 0.2,
}

# Bug-fix keyword regex (case-insensitive). Used on commit messages.
# Based on Mockus & Votta 2000 and Fischer et al. 2003.
BUG_FIX_KEYWORDS = (
    r"\bfix(?:es|ed|ing)?\b",
    r"\bbug(?:s|fix|fixes)?\b",
    r"\bdefect(?:s)?\b",
    r"\berror(?:s)?\b",
    r"\bpatch(?:es|ed)?\b",
    r"\bresolve(?:d|s)?\b",
    r"\bissue\s*#?\d+",
    r"\bclose(?:s|d)?\s*#?\d+",
)

# Jira issue-key regex to extract links from commit messages (e.g. "HBASE-1234").
JIRA_ISSUE_KEY_PATTERN = r"\b([A-Z][A-Z0-9_]+)-(\d+)\b"

# Severity levels considered "high-risk" for the severity baseline variant.
SEVERITY_BASELINE_LEVELS = ("BLOCKER", "CRITICAL")

# ---------------------------------------------------------------------------
# 5. Feature catalogues (Proposal Table 1)
# ---------------------------------------------------------------------------
# Static code metrics extracted from SONAR_MEASURES at snapshot time.
STATIC_METRICS = [
    "ncloc",
    "complexity",
    "cognitive_complexity",
    "classes",
    "functions",
    "statements",
    "duplicated_lines_density",
    "coverage",
    "comment_lines_density",
    "sqale_index",
    "sqale_debt_ratio",
    "file_complexity",
]

# Derived static features computed from the raw metrics above.
STATIC_DERIVED_FEATURES = [
    "cyclomatic_density",            # complexity / ncloc
    "comment_to_code_ratio",         # comment_lines_density-based
]

# Rule-violation count features from SONAR_ISSUES (at snapshot time).
# These are used as predictors for the consequence and SZZ variants, and
# EXCLUDED for the severity variant to avoid leakage.
ISSUE_COUNT_FEATURES = [
    "code_smells_nonsevere",
    "bugs_nonsevere",
    "vulnerabilities_nonsevere",
    "major_issues",
    "minor_issues",
    "info_issues",
]

# Features to DROP when training on severity-baseline labels to prevent
# label leakage. The severity label is ``(n_blocker + n_critical) > 0``
# (per ``SEVERITY_BASELINE_LEVELS``); any feature that carries that signal
# at inference time must be removed.
SEVERITY_LEAKY_FEATURES = [
    "n_blocker",
    "n_critical",
    "n_major",
    "n_minor",
    "n_info",
    "max_severity_rank",
]

# Features to DROP when training on SZZ-baseline labels. The SZZ label is
# ``(n_szz_fixes_future > 0)``; ``n_szz_inducing_past`` carries strongly
# correlated signal (files previously caught by SZZ are usually caught
# again), so we drop it to keep SZZ predictions non-trivial.
SZZ_LEAKY_FEATURES: list[str] = []  # features module currently carries no SZZ features

# Historical metrics from GIT_COMMITS and GIT_COMMITS_CHANGES up to snapshot t.
# Note: TD Dataset v2.0 does not expose a change-type column, so ADD / MODIFY /
# DELETE counts are replaced with distributional churn statistics (max, std)
# that capture volatility without requiring change-type annotations.
HISTORICAL_METRICS = [
    "total_commits_pre",
    "total_contributors_pre",
    "code_added_pre",
    "code_removed_pre",
    "code_churn_pre",
    "recent_churn_30d_pre",
    "recent_churn_90d_pre",
    "recent_commits_30d_pre",
    "recent_commits_90d_pre",
    "file_age_days_at_snapshot",
    "days_since_last_change_at_snapshot",
    "ownership_ratio_pre",
    "avg_change_size_pre",
    "max_single_commit_churn_pre",
    "std_change_size_pre",
]

# ---------------------------------------------------------------------------
# 6. Modeling
# ---------------------------------------------------------------------------
RANDOM_STATE = 42
TEST_SIZE = 0.2

MODELS = {
    "decision_tree": {
        "class": "DecisionTreeClassifier",
        "params": {"random_state": RANDOM_STATE, "class_weight": "balanced"},
    },
    "random_forest": {
        "class": "RandomForestClassifier",
        "params": {
            "n_estimators": 200,
            "random_state": RANDOM_STATE,
            "class_weight": "balanced",
            "n_jobs": -1,
        },
    },
    "svm": {
        "class": "SVC",
        "params": {
            "kernel": "rbf",
            "probability": True,
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
        },
    },
    "xgboost": {
        "class": "XGBClassifier",
        "params": {
            "n_estimators": 200,
            "random_state": RANDOM_STATE,
            "use_label_encoder": False,
            "eval_metric": "logloss",
            "n_jobs": -1,
        },
    },
    "lightgbm": {
        "class": "LGBMClassifier",
        "params": {
            "n_estimators": 200,
            "random_state": RANDOM_STATE,
            "class_weight": "balanced",
            "n_jobs": -1,
            "verbose": -1,
        },
    },
}

PARAM_GRIDS = {
    "xgboost": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7, 10],
        "learning_rate": [0.01, 0.1, 0.2],
        "scale_pos_weight": [1, 3, 5, 10],
    },
    "random_forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [10, 20, 30, None],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
    },
    "lightgbm": {
        "n_estimators": [100, 200, 300],
        "max_depth": [-1, 5, 10, 20],
        "learning_rate": [0.01, 0.1, 0.2],
        "num_leaves": [15, 31, 63],
    },
}

# ---------------------------------------------------------------------------
# 7. Evaluation
# ---------------------------------------------------------------------------
CV_FOLDS = 10                       # within-project stratified K-fold
COST_EFFECTIVENESS_AT = 0.20        # CE @ top-20% (prioritization metric)

# Variant keys used across pipeline scripts.
LABEL_VARIANTS = ("consequence", "severity", "szz")


<a id="source-modules"></a>
## Source modules

The next 14 cells materialise the production code modules under `/content/src/` using `%%writefile`. Module contents are *verbatim* copies of the repository sources - they are the same code that the local `scripts/01..10` use, ensuring byte-identical experimental results.

Each cell is preceded by a one-line caption stating the file path and purpose. After all 14 modules are written, a final cell imports them so subsequent stage cells can call them directly.


**`src/data/load_data.py`** - Thin SQL readers for the SQLite dataset.

In [ ]:
%%writefile /content/src/data/load_data.py
"""
Thin SQL readers for the Technical Debt Dataset v2.0.

This module contains ONLY raw SELECT helpers - no cleaning, no aggregation,
no domain logic. Cleaning lives in ``src.data.clean`` and labeling in
``src.data.labeling``.

Every function accepts an optional ``sqlite3.Connection``; callers that open
many queries should share one connection to avoid file-open overhead on a
1.5 GB database.

References
----------
Lenarduzzi, V., Saarimaki, N., Taibi, D. (2019). The Technical Debt Dataset.
Proc. PROMISE 2019.
"""
from __future__ import annotations

import sqlite3
import sys
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import TD_DATASET_PATH  # noqa: E402


# ---------------------------------------------------------------------------
# Connection helpers
# ---------------------------------------------------------------------------
def get_connection(db_path: Optional[Path] = None) -> sqlite3.Connection:
    """Open a SQLite connection to the Technical Debt Dataset."""
    db_path = Path(db_path) if db_path else TD_DATASET_PATH
    if not db_path.exists():
        raise FileNotFoundError(
            f"Database not found at {db_path}\n"
            f"Download: https://github.com/clowee/The-Technical-Debt-Dataset/releases"
        )
    return sqlite3.connect(str(db_path))


def get_table_names(conn: sqlite3.Connection) -> list[str]:
    """List all tables in the database."""
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;")
    return [r[0] for r in cur.fetchall()]


def get_table_schema(conn: sqlite3.Connection, table: str) -> pd.DataFrame:
    """Return PRAGMA table_info rows for a table."""
    return pd.read_sql_query(f"PRAGMA table_info({table});", conn)


def get_row_count(conn: sqlite3.Connection, table: str) -> int:
    """Return COUNT(*) of a table."""
    cur = conn.cursor()
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    return cur.fetchone()[0]


def get_sample_rows(conn: sqlite3.Connection, table: str, n: int = 3) -> pd.DataFrame:
    """Return the first ``n`` rows of a table."""
    return pd.read_sql_query(f"SELECT * FROM {table} LIMIT {int(n)};", conn)


# ---------------------------------------------------------------------------
# Table-specific loaders
# ---------------------------------------------------------------------------
def list_projects(conn: sqlite3.Connection) -> list[str]:
    """Return the sorted list of distinct project IDs using GIT_COMMITS."""
    try:
        rows = pd.read_sql_query(
            "SELECT DISTINCT PROJECT_ID FROM GIT_COMMITS ORDER BY PROJECT_ID",
            conn,
        )
    except Exception:
        rows = pd.read_sql_query(
            "SELECT DISTINCT PROJECT_ID FROM SONAR_ISSUES ORDER BY PROJECT_ID",
            conn,
        )
    return rows["PROJECT_ID"].dropna().astype(str).tolist()


def load_git_commits(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
    main_branch_only: bool = True,
    columns: Optional[list[str]] = None,
) -> pd.DataFrame:
    """Load GIT_COMMITS, optionally restricted to projects / main branch.

    Column names verified in Stage 1: ``COMMIT_HASH``, ``COMMIT_MESSAGE``,
    ``AUTHOR_DATE``, ``COMMITTER_DATE``, ``IN_MAIN_BRANCH`` (stored as text
    strings ``'True'``/``'False'``).
    """
    cols_expr = ", ".join(columns) if columns else "*"
    sql = f"SELECT {cols_expr} FROM GIT_COMMITS"
    where = []
    if main_branch_only:
        where.append("IN_MAIN_BRANCH = 'True'")
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        where.append(f"PROJECT_ID IN ({ids})")
    if where:
        sql += " WHERE " + " AND ".join(where)
    df = pd.read_sql_query(sql, conn)
    if "AUTHOR_DATE" in df.columns:
        df["AUTHOR_DATE"] = pd.to_datetime(df["AUTHOR_DATE"], errors="coerce", utc=True)
    if "COMMITTER_DATE" in df.columns:
        df["COMMITTER_DATE"] = pd.to_datetime(df["COMMITTER_DATE"], errors="coerce", utc=True)
    return df


def load_git_commits_changes(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Load GIT_COMMITS_CHANGES, optionally restricted to projects."""
    sql = "SELECT * FROM GIT_COMMITS_CHANGES"
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        sql += f" WHERE PROJECT_ID IN ({ids})"
    return pd.read_sql_query(sql, conn)


def load_sonar_measures(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Load SONAR_MEASURES; schema is wide (one column per metric)."""
    sql = "SELECT * FROM SONAR_MEASURES"
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        sql += f" WHERE PROJECT_ID IN ({ids})"
    return pd.read_sql_query(sql, conn)


def load_sonar_issues(
    conn: sqlite3.Connection,
    projects: Optional[Iterable[str]] = None,
) -> pd.DataFrame:
    """Load SONAR_ISSUES."""
    sql = "SELECT * FROM SONAR_ISSUES"
    if projects:
        ids = ",".join(f"'{p}'" for p in projects)
        sql += f" WHERE PROJECT_ID IN ({ids})"
    return pd.read_sql_query(sql, conn)


def load_jira_issues(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load the JIRA_ISSUES table. Schema is confirmed by Stage 1."""
    return pd.read_sql_query("SELECT * FROM JIRA_ISSUES", conn)


def load_szz_fault_inducing(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load SZZ fault-inducing / fault-fixing commit links.

    Columns verified in Stage 1: ``PROJECT_ID``, ``FAULT_FIXING_COMMIT_HASH``,
    ``FAULT_INDUCING_COMMIT_HASH``.
    """
    return pd.read_sql_query("SELECT * FROM SZZ_FAULT_INDUCING_COMMITS", conn)


def load_sonar_analysis(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load SONAR_ANALYSIS (maps ANALYSIS_KEY to snapshot DATE + Git REVISION).

    Columns verified in Stage 1: ``PROJECT_ID``, ``ANALYSIS_KEY``, ``DATE``,
    ``REVISION``. This table is the bridge between ``SONAR_MEASURES`` snapshots
    and Git history (via REVISION hash and DATE).
    """
    df = pd.read_sql_query("SELECT * FROM SONAR_ANALYSIS", conn)
    if "DATE" in df.columns:
        df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce", utc=True)
    return df


def load_projects_meta(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load the PROJECTS metadata table (31 rows, PROJECT_ID + GIT_LINK + JIRA_LINK)."""
    return pd.read_sql_query("SELECT * FROM PROJECTS", conn)


def load_refactorings(conn: sqlite3.Connection) -> pd.DataFrame:
    """Load refactoring records (optional qualitative validation)."""
    try:
        return pd.read_sql_query("SELECT * FROM REFACTORING_MINER", conn)
    except Exception:
        return pd.read_sql_query("SELECT * FROM REFACTORINGS", conn)


# ---------------------------------------------------------------------------
# Windowed helpers
# ---------------------------------------------------------------------------
def commits_in_window(
    conn: sqlite3.Connection,
    project_id: str,
    start: pd.Timestamp,
    end: pd.Timestamp,
    main_branch_only: bool = True,
) -> pd.DataFrame:
    """Return commits in the half-open window (start, end] for a project."""
    parts = ["PROJECT_ID = ?", "AUTHOR_DATE > ?", "AUTHOR_DATE <= ?"]
    params: list = [project_id, start.isoformat(), end.isoformat()]
    if main_branch_only:
        parts.append("IN_MAIN_BRANCH = 'True'")
    sql = "SELECT * FROM GIT_COMMITS WHERE " + " AND ".join(parts)
    df = pd.read_sql_query(sql, conn, params=params)
    if "AUTHOR_DATE" in df.columns:
        df["AUTHOR_DATE"] = pd.to_datetime(df["AUTHOR_DATE"], errors="coerce", utc=True)
    return df


**`src/data/snapshot.py`** - Per-project snapshot date selection.

In [ ]:
%%writefile /content/src/data/snapshot.py
"""
Snapshot date selection and temporal-split utilities.

Implements the temporal-split protocol described in the approved proposal
Section 3.4: for each project, a snapshot time ``t`` is selected such that
features are computed from commits with ``AUTHOR_DATE <= t`` and labels are
derived from commits in the observation window ``(t, t + W]`` where ``W``
is the observation window (6 months primary).

The default strategy is the **median master-branch commit date** per project,
which guarantees every project has both a meaningful pre-snapshot history
(for features) and a meaningful post-snapshot window (for labels).

References
----------
- Zimmermann, T., Nagappan, N. (2008). Predicting defects using network analysis
  on dependency graphs. ICSE 2008 - basis for 6-month observation window.
- Jiang, Z., Chen, T., Zhou, Y. (2024). Improving technical debt prediction with
  graph-based and social-network metrics. Empir. Softw. Eng. 29 - uses median
  snapshot strategy.
"""
from __future__ import annotations

import sqlite3
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import (  # noqa: E402
    MIN_POST_SNAPSHOT_COMMITS,
    MIN_PRE_SNAPSHOT_COMMITS,
    OBSERVATION_WINDOW_MONTHS,
    SNAPSHOT_STRATEGY,
)


@dataclass
class ProjectSnapshot:
    """Per-project snapshot metadata."""

    project_id: str
    snapshot_date: pd.Timestamp
    window_months: int
    first_commit: pd.Timestamp
    last_commit: pd.Timestamp
    total_commits: int
    pre_snapshot_commits: int
    post_snapshot_commits: int
    distinct_files_pre: int
    distinct_authors_pre: int
    eligible: bool
    exclusion_reason: str = ""

    def to_dict(self) -> dict:
        return {
            "project_id": self.project_id,
            "snapshot_date": self.snapshot_date,
            "window_months": self.window_months,
            "window_end": self.window_end,
            "first_commit": self.first_commit,
            "last_commit": self.last_commit,
            "total_commits": self.total_commits,
            "pre_snapshot_commits": self.pre_snapshot_commits,
            "post_snapshot_commits": self.post_snapshot_commits,
            "distinct_files_pre": self.distinct_files_pre,
            "distinct_authors_pre": self.distinct_authors_pre,
            "eligible": self.eligible,
            "exclusion_reason": self.exclusion_reason,
        }

    @property
    def window_end(self) -> pd.Timestamp:
        return self.snapshot_date + pd.DateOffset(months=self.window_months)


def _add_months(t: pd.Timestamp, months: int) -> pd.Timestamp:
    return t + pd.DateOffset(months=months)


def compute_project_snapshot(
    conn: sqlite3.Connection,
    project_id: str,
    strategy: str = SNAPSHOT_STRATEGY,
    window_months: int = OBSERVATION_WINDOW_MONTHS,
    min_pre: int = MIN_PRE_SNAPSHOT_COMMITS,
    min_post: int = MIN_POST_SNAPSHOT_COMMITS,
) -> ProjectSnapshot:
    """Compute the snapshot date for a single project.

    Parameters
    ----------
    conn :
        SQLite connection to the Technical Debt Dataset.
    project_id :
        PROJECT_ID value (e.g. ``'org.apache:batik'``).
    strategy :
        One of ``'median'`` (default), ``'fixed'`` (uses ``window_months`` back
        from the last commit), or ``'release'`` (not implemented yet).
    window_months :
        Length of the post-snapshot observation window.
    min_pre, min_post :
        Minimum number of pre- and post-snapshot commits required for a
        project to be considered eligible.
    """
    query = (
        "SELECT AUTHOR_DATE, COMMIT_HASH, AUTHOR "
        "FROM GIT_COMMITS "
        "WHERE PROJECT_ID = ? AND IN_MAIN_BRANCH = 'True' "
        "ORDER BY AUTHOR_DATE ASC"
    )
    commits = pd.read_sql_query(query, conn, params=[project_id])
    commits["AUTHOR_DATE"] = pd.to_datetime(commits["AUTHOR_DATE"], errors="coerce", utc=True)
    commits = commits.dropna(subset=["AUTHOR_DATE"])

    if len(commits) == 0:
        return ProjectSnapshot(
            project_id=project_id,
            snapshot_date=pd.NaT,
            window_months=window_months,
            first_commit=pd.NaT,
            last_commit=pd.NaT,
            total_commits=0,
            pre_snapshot_commits=0,
            post_snapshot_commits=0,
            distinct_files_pre=0,
            distinct_authors_pre=0,
            eligible=False,
            exclusion_reason="no_commits",
        )

    first = commits["AUTHOR_DATE"].min()
    last = commits["AUTHOR_DATE"].max()

    if strategy == "median":
        snapshot = commits["AUTHOR_DATE"].quantile(0.5, interpolation="nearest")
    elif strategy == "fixed":
        snapshot = _add_months(last, -window_months)
    elif strategy == "release":
        raise NotImplementedError("release-tag strategy not implemented yet")
    else:
        raise ValueError(f"Unknown snapshot strategy: {strategy}")

    snapshot = pd.Timestamp(snapshot)
    window_end = _add_months(snapshot, window_months)

    pre_mask = commits["AUTHOR_DATE"] <= snapshot
    post_mask = (commits["AUTHOR_DATE"] > snapshot) & (commits["AUTHOR_DATE"] <= window_end)
    pre_count = int(pre_mask.sum())
    post_count = int(post_mask.sum())

    # Distinct pre-snapshot files / authors for profiling
    if pre_count > 0:
        pre_hashes = commits.loc[pre_mask, "COMMIT_HASH"].tolist()
        if len(pre_hashes) > 0:
            placeholders = ",".join("?" * len(pre_hashes))
            file_q = (
                f"SELECT COUNT(DISTINCT FILE) AS n "
                f"FROM GIT_COMMITS_CHANGES "
                f"WHERE PROJECT_ID = ? AND COMMIT_HASH IN ({placeholders})"
            )
            distinct_files = pd.read_sql_query(file_q, conn, params=[project_id, *pre_hashes]).iloc[0]["n"]
        else:
            distinct_files = 0
        distinct_authors = int(commits.loc[pre_mask, "AUTHOR"].nunique())
    else:
        distinct_files = 0
        distinct_authors = 0

    eligible = pre_count >= min_pre and post_count >= min_post
    reason = ""
    if not eligible:
        reasons = []
        if pre_count < min_pre:
            reasons.append(f"pre<{min_pre}")
        if post_count < min_post:
            reasons.append(f"post<{min_post}")
        reason = ",".join(reasons)

    return ProjectSnapshot(
        project_id=project_id,
        snapshot_date=snapshot,
        window_months=window_months,
        first_commit=pd.Timestamp(first),
        last_commit=pd.Timestamp(last),
        total_commits=int(len(commits)),
        pre_snapshot_commits=pre_count,
        post_snapshot_commits=post_count,
        distinct_files_pre=int(distinct_files),
        distinct_authors_pre=int(distinct_authors),
        eligible=bool(eligible),
        exclusion_reason=reason,
    )


def compute_all_snapshots(
    conn: sqlite3.Connection,
    projects: Optional[list[str]] = None,
    strategy: str = SNAPSHOT_STRATEGY,
    window_months: int = OBSERVATION_WINDOW_MONTHS,
) -> pd.DataFrame:
    """Compute snapshots for every project in the dataset (or a given list)."""
    if projects is None:
        cur = conn.cursor()
        cur.execute("SELECT DISTINCT PROJECT_ID FROM GIT_COMMITS ORDER BY PROJECT_ID")
        projects = [r[0] for r in cur.fetchall()]

    rows: list[dict] = []
    for pid in projects:
        snap = compute_project_snapshot(conn, pid, strategy, window_months)
        rows.append(snap.to_dict())
    return pd.DataFrame(rows)


def load_snapshots(path: Path) -> pd.DataFrame:
    """Load previously computed snapshots from disk (Parquet)."""
    df = pd.read_parquet(path)
    for col in ("snapshot_date", "window_end", "first_commit", "last_commit"):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], utc=True, errors="coerce")
    return df


**`src/data/clean.py`** - Path normalisation, bug-fix tagging, type coercion.

In [ ]:
%%writefile /content/src/data/clean.py
"""
Data cleaning and normalization for the Technical Debt Dataset v2.0.

This module converts raw SQLite tables into tidy pandas DataFrames suitable
for feature extraction (Stage 5), labeling (Stage 4) and cross-table joins
(Stage 6). The main transformations are:

1. **Path normalization** - ``SONAR_ISSUES.COMPONENT`` stores file identifiers
   as ``SonarProjectKey:path/to/File.java`` whereas ``GIT_COMMITS_CHANGES.FILE``
   stores them as plain repo-relative paths. We strip the prefix so the two
   tables join cleanly on ``(PROJECT_ID, file_path)``.
2. **Temporal enrichment** - ``SONAR_MEASURES`` rows carry an analysis key
   but not a date. We join ``SONAR_ANALYSIS`` to attach ``DATE`` and
   ``REVISION`` (Git hash of the analyzed commit) so measures can be
   temporally aligned to the snapshot ``t``.
3. **Bug-fix tagging** - regex-based flagging of commit messages plus a
   Jira-link flag derived from the pre-populated ``JIRA_ISSUES.HASH``
   column and Jira key mentions in commit messages.
4. **Scope filtering** - restricts to Java source files (excluding
   tests, generated sources, build artefacts) per ``config.py``.
5. **Project filtering** - keeps only projects flagged *eligible* by
   Stage 2.
6. **Type coercion** - parses dates to tz-aware ``datetime64[ns, UTC]``,
   casts numeric columns, removes obvious duplicates.

Each cleaned DataFrame is persisted as Parquet in ``data/processed/``.

References
----------
- Lenarduzzi, V., et al. (2019). The Technical Debt Dataset. PROMISE 2019.
- Mockus, A., Votta, L. (2000). Identifying reasons for software changes
  using historic databases. ICSM 2000 - bug-fix keyword regex basis.
"""
from __future__ import annotations

import re
import sqlite3
import sys
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import (  # noqa: E402
    BUG_FIX_KEYWORDS,
    JIRA_ISSUE_KEY_PATTERN,
    PATH_EXCLUSION_PATTERNS,
    SOURCE_FILE_EXTENSIONS,
)


_BUGFIX_REGEX = re.compile("|".join(BUG_FIX_KEYWORDS), re.IGNORECASE)
_JIRA_KEY_REGEX = re.compile(JIRA_ISSUE_KEY_PATTERN)


# ---------------------------------------------------------------------------
# Path normalization
# ---------------------------------------------------------------------------
def normalize_component_path(component: Optional[str]) -> Optional[str]:
    """Strip the ``SonarProjectKey:`` prefix from ``SONAR_ISSUES.COMPONENT``.

    Returns the repo-relative file path, e.g.
    ``"Apache_Cayenne:framework/.../Fault.java"`` -> ``"framework/.../Fault.java"``.
    Components without a colon are returned unchanged. ``None`` values are
    preserved.
    """
    if component is None or not isinstance(component, str):
        return component
    idx = component.find(":")
    if idx < 0:
        return component
    return component[idx + 1 :]


def is_java_source(path: Optional[str]) -> bool:
    """True if ``path`` is a non-test, non-generated Java source file."""
    if path is None or not isinstance(path, str) or not path:
        return False
    p = path.replace("\\", "/").lower()
    if not any(p.endswith(ext) for ext in SOURCE_FILE_EXTENSIONS):
        return False
    return not any(bad in p for bad in PATH_EXCLUSION_PATTERNS)


def extract_basename(path: Optional[str]) -> Optional[str]:
    """Return the final segment of a slash- or backslash-separated path.

    This is the canonical file-identifier key used across the pipeline because
    ``GIT_COMMITS_CHANGES.FILE`` in TD Dataset v2.0 stores only the basename
    for most projects (see RESEARCH_LOG.md 2026-04-23 path-format entry).
    """
    if path is None or not isinstance(path, str) or not path:
        return path
    p = path.replace("\\", "/")
    return p.rsplit("/", 1)[-1]


# ---------------------------------------------------------------------------
# Commit tagging
# ---------------------------------------------------------------------------
def flag_bugfix_commits(messages: pd.Series) -> pd.Series:
    """Return a boolean Series marking messages that look like bug-fixes.

    Uses the union regex of ``BUG_FIX_KEYWORDS`` (Mockus & Votta 2000).
    Empty / NaN messages return ``False``.
    """
    return messages.fillna("").astype(str).str.contains(_BUGFIX_REGEX, regex=True)


def extract_jira_keys(messages: pd.Series) -> pd.Series:
    """Return a Series of lists of Jira keys mentioned in each commit message."""

    def _find(msg: str) -> list[str]:
        if not isinstance(msg, str) or not msg:
            return []
        return [f"{m.group(1)}-{m.group(2)}" for m in _JIRA_KEY_REGEX.finditer(msg)]

    return messages.apply(_find)


# ---------------------------------------------------------------------------
# Per-table cleaners
# ---------------------------------------------------------------------------
def clean_git_commits(
    conn: sqlite3.Connection,
    projects: Iterable[str],
) -> pd.DataFrame:
    """Load master-branch commits for ``projects`` and add cleaning columns.

    Columns added:
    - ``AUTHOR_DATE`` parsed to UTC-aware datetime
    - ``is_bugfix`` (bool)
    - ``jira_keys`` (list[str])
    """
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, COMMIT_HASH, COMMIT_MESSAGE, AUTHOR, AUTHOR_DATE, "
        "       COMMITTER, COMMITTER_DATE, MERGE "
        "FROM GIT_COMMITS "
        f"WHERE IN_MAIN_BRANCH = 'True' AND PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df["AUTHOR_DATE"] = pd.to_datetime(df["AUTHOR_DATE"], errors="coerce", utc=True)
    df["COMMITTER_DATE"] = pd.to_datetime(df["COMMITTER_DATE"], errors="coerce", utc=True)
    df = df.dropna(subset=["AUTHOR_DATE", "COMMIT_HASH"]).copy()
    df["is_bugfix"] = flag_bugfix_commits(df["COMMIT_MESSAGE"])
    df["jira_keys"] = extract_jira_keys(df["COMMIT_MESSAGE"])
    df["MERGE"] = df["MERGE"].astype(str).str.lower().isin(("true", "1"))
    df = df.drop_duplicates(subset=["PROJECT_ID", "COMMIT_HASH"])
    return df.reset_index(drop=True)


def clean_git_commits_changes(
    conn: sqlite3.Connection,
    projects: Iterable[str],
    java_only: bool = True,
) -> pd.DataFrame:
    """Load per-file changes for ``projects``; optionally keep only Java sources.

    Output columns: ``PROJECT_ID``, ``COMMIT_HASH``, ``file_path`` (normalized),
    ``DATE`` (UTC), ``COMMITTER_ID``, ``LINES_ADDED``, ``LINES_REMOVED``.
    """
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, COMMIT_HASH, FILE, DATE, COMMITTER_ID, "
        "       LINES_ADDED, LINES_REMOVED "
        "FROM GIT_COMMITS_CHANGES "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df = df.rename(columns={"FILE": "file_path"})
    df["file_path"] = df["file_path"].astype(str).str.replace("\\", "/", regex=False)
    df["DATE"] = pd.to_datetime(df["DATE"], errors="coerce", utc=True)
    for c in ("LINES_ADDED", "LINES_REMOVED"):
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype("int64")
    df = df.dropna(subset=["file_path", "COMMIT_HASH", "DATE"])
    df = df[df["file_path"].str.len() > 0]
    if java_only:
        df = df[df["file_path"].apply(is_java_source)]
    df["basename"] = df["file_path"].apply(extract_basename)
    df = df.drop_duplicates(subset=["PROJECT_ID", "COMMIT_HASH", "file_path"])
    return df.reset_index(drop=True)


def clean_sonar_issues(
    conn: sqlite3.Connection,
    projects: Iterable[str],
    java_only: bool = True,
) -> pd.DataFrame:
    """Load SONAR_ISSUES, normalize COMPONENT to ``file_path``, coerce types."""
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, ISSUE_KEY, TYPE, RULE, SEVERITY, STATUS, RESOLUTION, "
        "       EFFORT, DEBT, CREATION_DATE, CLOSE_DATE, COMPONENT, "
        "       START_LINE, END_LINE "
        "FROM SONAR_ISSUES "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df["file_path"] = df["COMPONENT"].apply(normalize_component_path)
    df["file_path"] = df["file_path"].astype(str).str.replace("\\", "/", regex=False)
    df["CREATION_DATE"] = pd.to_datetime(df["CREATION_DATE"], errors="coerce", utc=True)
    df["CLOSE_DATE"] = pd.to_datetime(df["CLOSE_DATE"], errors="coerce", utc=True)
    for c in ("EFFORT", "DEBT", "START_LINE", "END_LINE"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["file_path", "ISSUE_KEY"])
    df = df[df["file_path"].str.len() > 0]
    if java_only:
        df = df[df["file_path"].apply(is_java_source)]
    df["basename"] = df["file_path"].apply(extract_basename)
    return df.drop(columns=["COMPONENT"]).reset_index(drop=True)


def clean_sonar_measures_with_dates(
    conn: sqlite3.Connection,
    projects: Iterable[str],
) -> pd.DataFrame:
    """Load SONAR_MEASURES and join SONAR_ANALYSIS to attach DATE + REVISION.

    Returns one row per ``(PROJECT_ID, ANALYSIS_KEY)`` with ``analysis_date``
    (UTC-aware) and the canonical subset of project-level metrics used as
    features (NCLOC, COMPLEXITY, SQALE_INDEX, ...). Leaky raw severity counts
    are kept here because they will be selectively dropped at dataset-build
    time depending on the label variant.
    """
    ids = ",".join(f"'{p}'" for p in projects)
    metric_cols = [
        "NCLOC",
        "LINES",
        "CLASSES",
        "FILES",
        "FUNCTIONS",
        "STATEMENTS",
        "COMPLEXITY",
        "COGNITIVE_COMPLEXITY",
        "FILE_COMPLEXITY",
        "FUNCTION_COMPLEXITY",
        "CLASS_COMPLEXITY",
        "COMMENT_LINES",
        "COMMENT_LINES_DENSITY",
        "DUPLICATED_LINES",
        "DUPLICATED_LINES_DENSITY",
        "DUPLICATED_BLOCKS",
        "DUPLICATED_FILES",
        "COVERAGE",
        "LINE_COVERAGE",
        "LINES_TO_COVER",
        "UNCOVERED_LINES",
        "VIOLATIONS",
        "BLOCKER_VIOLATIONS",
        "CRITICAL_VIOLATIONS",
        "MAJOR_VIOLATIONS",
        "MINOR_VIOLATIONS",
        "INFO_VIOLATIONS",
        "CODE_SMELLS",
        "BUGS",
        "VULNERABILITIES",
        "SQALE_INDEX",
        "SQALE_DEBT_RATIO",
        "SQALE_RATING",
        "RELIABILITY_RATING",
        "SECURITY_RATING",
        "RELIABILITY_REMEDIATION_EFFORT",
        "SECURITY_REMEDIATION_EFFORT",
        "OPEN_ISSUES",
    ]
    cols_sql = ", ".join(f"m.{c}" for c in metric_cols)
    sql = (
        f"SELECT m.PROJECT_ID, m.ANALYSIS_KEY, a.DATE AS analysis_date, "
        f"       a.REVISION AS analysis_revision, {cols_sql} "
        "FROM SONAR_MEASURES m "
        "LEFT JOIN SONAR_ANALYSIS a "
        "       ON a.PROJECT_ID = m.PROJECT_ID "
        "      AND a.ANALYSIS_KEY = m.ANALYSIS_KEY "
        f"WHERE m.PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    df["analysis_date"] = pd.to_datetime(df["analysis_date"], errors="coerce", utc=True)
    for c in metric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["analysis_date"]).copy()
    df.columns = [c.lower() if c.isupper() else c for c in df.columns]
    return df.sort_values(["project_id", "analysis_date"]).reset_index(drop=True)


def clean_szz_with_dates(
    conn: sqlite3.Connection,
    projects: Iterable[str],
    git_commits_clean: pd.DataFrame,
) -> pd.DataFrame:
    """Load SZZ links and enrich with fault-fixing and fault-inducing dates.

    Joins to ``git_commits_clean`` on ``(PROJECT_ID, COMMIT_HASH)`` to attach
    ``AUTHOR_DATE`` for both sides of the pair.
    """
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, FAULT_FIXING_COMMIT_HASH, FAULT_INDUCING_COMMIT_HASH "
        "FROM SZZ_FAULT_INDUCING_COMMITS "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)

    commits_small = git_commits_clean[["PROJECT_ID", "COMMIT_HASH", "AUTHOR_DATE"]]
    df = df.merge(
        commits_small.rename(
            columns={"COMMIT_HASH": "FAULT_FIXING_COMMIT_HASH", "AUTHOR_DATE": "fix_date"}
        ),
        on=["PROJECT_ID", "FAULT_FIXING_COMMIT_HASH"],
        how="left",
    )
    df = df.merge(
        commits_small.rename(
            columns={"COMMIT_HASH": "FAULT_INDUCING_COMMIT_HASH", "AUTHOR_DATE": "induce_date"}
        ),
        on=["PROJECT_ID", "FAULT_INDUCING_COMMIT_HASH"],
        how="left",
    )
    return df.drop_duplicates().reset_index(drop=True)


def clean_jira_issues(
    conn: sqlite3.Connection,
    projects: Iterable[str],
) -> pd.DataFrame:
    """Load JIRA_ISSUES for projects with dates parsed and a ``is_bug`` flag."""
    ids = ",".join(f"'{p}'" for p in projects)
    sql = (
        "SELECT PROJECT_ID, KEY, PRIORITY, TYPE, STATUS, RESOLUTION, "
        "       CREATION_DATE, RESOLUTION_DATE, UPDATE_DATE, HASH, COMMIT_DATE "
        "FROM JIRA_ISSUES "
        f"WHERE PROJECT_ID IN ({ids})"
    )
    df = pd.read_sql_query(sql, conn)
    for c in ("CREATION_DATE", "RESOLUTION_DATE", "UPDATE_DATE", "COMMIT_DATE"):
        df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)
    df["is_bug"] = df["TYPE"].fillna("").str.lower().eq("bug")
    return df.reset_index(drop=True)


**`src/data/szz.py`** - SZZ / bug-fix / Jira helpers at basename granularity.

In [ ]:
%%writefile /content/src/data/szz.py
"""
SZZ, bug-fix and Jira resolution helpers operating on cleaned parquets.

These helpers bridge commit-level tables (GIT_COMMITS, GIT_COMMITS_CHANGES,
SZZ_FAULT_INDUCING_COMMITS, JIRA_ISSUES) and the basename-level units of
analysis used throughout the pipeline.

Key operations
--------------
- ``bugfix_touches_in_window`` - for a project, a snapshot ``t`` and a
  window length ``W``, return a DataFrame of ``(basename,
  n_bugfix_commits, churn_add, churn_removed)`` covering bug-fix commits
  whose ``AUTHOR_DATE`` falls in ``(t, t+W]``.
- ``churn_in_window`` - per-basename churn (sum of LINES_ADDED +
  LINES_REMOVED) in the post-snapshot window, irrespective of bug-fix
  flag. Used as the "future_churn" component of the consequence score.
- ``szz_events_in_window`` - per-basename count of SZZ fault-fixing
  commits in ``(t, t+W]`` that touched the basename (i.e. the fix
  modified the file). Used as the "szz_defects_future" component.
- ``jira_bug_commits_in_window`` - commits linked to Jira ``Bug`` issues
  that close inside the observation window. Used as a stricter bug-fix
  signal (sensitivity analysis; not in primary risk score to keep the
  score fully reproducible from the DB).

Everything is indexed by ``(project_id, basename)`` to match the chosen
unit of analysis (see RESEARCH_LOG.md 2026-04-23 entry).
"""
from __future__ import annotations

from typing import Optional

import pandas as pd


def _window_mask(dates: pd.Series, t: pd.Timestamp, t_end: pd.Timestamp) -> pd.Series:
    """Boolean mask for ``t < dates <= t_end`` with NaT-safe handling."""
    if dates.dtype.kind != "M":
        dates = pd.to_datetime(dates, errors="coerce", utc=True)
    return (dates > t) & (dates <= t_end)


def _ensure_utc(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    if ts.tz is None:
        ts = ts.tz_localize("UTC")
    return ts


def bugfix_touches_in_window(
    commits: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Bug-fix commit count and churn per basename in ``(t, t+W]``.

    Parameters
    ----------
    commits :
        ``clean_git_commits`` DataFrame with an ``is_bugfix`` column.
    changes :
        ``clean_git_commits_changes`` DataFrame with ``basename``.
    project_id :
        Which project to compute for.
    t :
        Snapshot timestamp (UTC-aware).
    window_months :
        Observation window length.

    Returns
    -------
    DataFrame with columns ``basename``, ``n_bugfix_commits_future``,
    ``bugfix_churn_future`` (added + removed lines in bug-fix commits).
    """
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    c = commits[commits["PROJECT_ID"] == project_id].copy()
    c = c[_window_mask(c["AUTHOR_DATE"], t, t_end) & c["is_bugfix"].fillna(False)]
    if c.empty:
        return pd.DataFrame(columns=["basename", "n_bugfix_commits_future", "bugfix_churn_future"])

    ch = changes[changes["PROJECT_ID"] == project_id].copy()
    ch = ch[ch["COMMIT_HASH"].isin(c["COMMIT_HASH"])]

    if ch.empty:
        return pd.DataFrame(columns=["basename", "n_bugfix_commits_future", "bugfix_churn_future"])

    ch = ch.assign(_row_churn=ch["LINES_ADDED"].astype("int64") + ch["LINES_REMOVED"].astype("int64"))
    agg = (
        ch.groupby("basename")
        .agg(
            n_bugfix_commits_future=("COMMIT_HASH", "nunique"),
            bugfix_churn_future=("_row_churn", "sum"),
        )
        .reset_index()
    )
    agg["bugfix_churn_future"] = agg["bugfix_churn_future"].astype("int64")
    return agg


def churn_in_window(
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Total churn per basename in ``(t, t+W]`` (all commits, not only bug-fixes)."""
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    ch = changes[changes["PROJECT_ID"] == project_id].copy()
    ch = ch[_window_mask(ch["DATE"], t, t_end)]
    if ch.empty:
        return pd.DataFrame(
            columns=["basename", "future_churn", "future_add", "future_removed", "future_commits"]
        )

    agg = (
        ch.groupby("basename")
        .agg(
            future_add=("LINES_ADDED", "sum"),
            future_removed=("LINES_REMOVED", "sum"),
            future_commits=("COMMIT_HASH", "nunique"),
        )
        .reset_index()
    )
    agg["future_churn"] = (agg["future_add"] + agg["future_removed"]).astype("int64")
    return agg[["basename", "future_churn", "future_add", "future_removed", "future_commits"]]


def szz_events_in_window(
    szz: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Per-basename count of SZZ fault-fixing commits in ``(t, t+W]`` touching it.

    Uses the ``fix_date`` attached in ``clean_szz_with_dates``. The file
    resolution is done by joining SZZ's ``FAULT_FIXING_COMMIT_HASH`` to
    ``GIT_COMMITS_CHANGES.COMMIT_HASH`` to find touched basenames.
    """
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    s = szz[szz["PROJECT_ID"] == project_id].copy()
    s = s[_window_mask(s["fix_date"], t, t_end)]
    if s.empty:
        return pd.DataFrame(columns=["basename", "n_szz_fixes_future", "n_szz_inducing_past"])

    ch = changes[changes["PROJECT_ID"] == project_id]

    # (1) Basenames touched by fault-fixing commits in window
    fix_hashes = set(s["FAULT_FIXING_COMMIT_HASH"].dropna().unique())
    fix_touches = ch[ch["COMMIT_HASH"].isin(fix_hashes)][["basename", "COMMIT_HASH"]]
    fix_agg = (
        fix_touches.groupby("basename")["COMMIT_HASH"]
        .nunique()
        .rename("n_szz_fixes_future")
        .reset_index()
    )

    # (2) Basenames that were originally modified by fault-inducing commits
    # before the snapshot (these are the "ticking-time-bomb" files)
    induce = s[s["induce_date"].notna() & (s["induce_date"] <= t)]
    induce_hashes = set(induce["FAULT_INDUCING_COMMIT_HASH"].dropna().unique())
    induce_touches = ch[ch["COMMIT_HASH"].isin(induce_hashes)][["basename", "COMMIT_HASH"]]
    induce_agg = (
        induce_touches.groupby("basename")["COMMIT_HASH"]
        .nunique()
        .rename("n_szz_inducing_past")
        .reset_index()
    )

    out = pd.merge(fix_agg, induce_agg, on="basename", how="outer").fillna(0)
    for c in ("n_szz_fixes_future", "n_szz_inducing_past"):
        out[c] = out[c].astype("int64")
    return out


def jira_bug_commits_in_window(
    commits: pd.DataFrame,
    jira: pd.DataFrame,
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
    window_months: int = 6,
) -> pd.DataFrame:
    """Per-basename count of commits that close a Jira *Bug* in ``(t, t+W]``.

    Uses the pre-populated ``JIRA_ISSUES.HASH`` column to link tickets to
    commits directly; supplements by scanning for Jira keys in commit
    messages of bug-fix commits. Returned as a separate diagnostic
    feature - not part of the primary risk score to keep the score
    computable for projects that lack rich Jira linkage.
    """
    t = _ensure_utc(t)
    t_end = t + pd.DateOffset(months=window_months)

    j = jira[jira["PROJECT_ID"] == project_id].copy()
    j = j[j["is_bug"]]
    bug_keys = set(j["KEY"].dropna().unique())
    hash_links = set(j["HASH"].dropna().unique())

    c = commits[commits["PROJECT_ID"] == project_id].copy()
    c = c[_window_mask(c["AUTHOR_DATE"], t, t_end)]
    if c.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bug_commits_future"])

    # Commits that either link to a bug ticket via hash OR mention a known bug key
    c["mentions_bug_key"] = c["jira_keys"].apply(
        lambda keys: any(k in bug_keys for k in keys) if isinstance(keys, list) else False
    )
    c["hash_in_bug_links"] = c["COMMIT_HASH"].isin(hash_links)
    c = c[c["mentions_bug_key"] | c["hash_in_bug_links"]]
    if c.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bug_commits_future"])

    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["COMMIT_HASH"].isin(c["COMMIT_HASH"])]
    if ch.empty:
        return pd.DataFrame(columns=["basename", "n_jira_bug_commits_future"])
    agg = (
        ch.groupby("basename")["COMMIT_HASH"]
        .nunique()
        .rename("n_jira_bug_commits_future")
        .reset_index()
    )
    return agg


def basename_universe_at_snapshot(
    changes: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
) -> pd.DataFrame:
    """Return the set of basenames that exist in ``project_id`` at time ``t``.

    A basename "exists at t" if any change with ``DATE <= t`` touched it.
    Output columns: ``project_id``, ``basename``.
    """
    t = _ensure_utc(t)
    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["DATE"] <= t]
    universe = ch[["basename"]].drop_duplicates().copy()
    universe["project_id"] = project_id
    return universe[["project_id", "basename"]].reset_index(drop=True)


def open_issues_at_snapshot(
    issues: pd.DataFrame,
    project_id: str,
    t: pd.Timestamp,
) -> pd.DataFrame:
    """Filter ``clean_sonar_issues`` to issues open at time ``t``.

    An issue is "open at ``t``" iff ``CREATION_DATE <= t`` and
    (``CLOSE_DATE`` is NaT or ``CLOSE_DATE > t``).
    """
    t = _ensure_utc(t)
    df = issues[issues["PROJECT_ID"] == project_id]
    creation_ok = df["CREATION_DATE"] <= t
    close_ok = df["CLOSE_DATE"].isna() | (df["CLOSE_DATE"] > t)
    return df[creation_ok & close_ok].copy()


**`src/data/labeling.py`** - Three-variant label computation + agreement metrics.

In [ ]:
%%writefile /content/src/data/labeling.py
"""
Three-variant labeling for High-Risk Technical Debt.

Implements the three mutually-disjoint label definitions compared in the
approved MSc proposal (Section 3.4):

1. **Consequence-oriented (primary)** - top ``P%`` per project by a weighted
   risk score combining (a) future bug-fix commits, (b) future churn and
   (c) future SZZ fault-inducing events in the observation window
   ``(t, t + W]``.
2. **Severity-based baseline** - positive iff the file has any SonarQube
   BLOCKER or CRITICAL issue open at snapshot ``t``. This is the
   conventional static-analysis view of "high-risk" debt.
3. **SZZ defect-oriented baseline** - positive iff the file is touched by
   at least one SZZ fault-inducing commit that is fixed inside the
   observation window (classical SZZ-defect-prediction target).

Each variant returns a DataFrame indexed by ``(project_id, basename)`` with
a binary ``is_high_risk`` column plus the numeric signals used to derive
it (kept for diagnostics and sensitivity analysis).

References
----------
- Lenarduzzi, V., et al. (2019). The Technical Debt Dataset. PROMISE.
- Kamei, Y., et al. (2013). A large-scale empirical study of just-in-time
  quality assurance. IEEE TSE 39(6).
- Tsoukalas, D., et al. (2020). Machine learning for technical debt
  identification. IEEE TSE.
- Jiang, Z., Chen, T., Zhou, Y. (2024). Graph-based technical debt
  prediction. Empir. Softw. Eng. 29.
"""
from __future__ import annotations

import sys
from pathlib import Path
from typing import Iterable, Optional

import numpy as np
import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import (  # noqa: E402
    HIGH_RISK_PERCENTILE,
    OBSERVATION_WINDOW_MONTHS,
    RISK_SCORE_WEIGHTS,
    SEVERITY_BASELINE_LEVELS,
)
from src.data.szz import (  # noqa: E402
    basename_universe_at_snapshot,
    bugfix_touches_in_window,
    churn_in_window,
    jira_bug_commits_in_window,
    open_issues_at_snapshot,
    szz_events_in_window,
)


# ---------------------------------------------------------------------------
# Utilities
# ---------------------------------------------------------------------------
def _min_max_norm(s: pd.Series) -> pd.Series:
    """Min-max normalize a numeric Series to ``[0, 1]``; constant series -> 0."""
    s = s.astype(float)
    lo, hi = s.min(), s.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi == lo:
        return pd.Series(np.zeros(len(s)), index=s.index)
    return (s - lo) / (hi - lo)


def _top_percentile(s: pd.Series, percentile: float) -> pd.Series:
    """Boolean mask for the top ``percentile``% values of ``s`` (ties broken by value).

    ``percentile`` is in ``[0, 100]``. If all values tie, returns all-False.
    """
    if len(s) == 0:
        return pd.Series([], dtype=bool)
    threshold = np.percentile(s, 100 - percentile)
    mask = s > threshold
    if mask.sum() == 0:
        mask = s >= threshold
    return mask


# ---------------------------------------------------------------------------
# Consequence-oriented (primary) labeling
# ---------------------------------------------------------------------------
def compute_consequence_labels(
    project_id: str,
    snapshot: pd.Timestamp,
    commits: pd.DataFrame,
    changes: pd.DataFrame,
    szz: pd.DataFrame,
    jira: Optional[pd.DataFrame] = None,
    window_months: int = OBSERVATION_WINDOW_MONTHS,
    percentile: float = HIGH_RISK_PERCENTILE,
    weights: dict = RISK_SCORE_WEIGHTS,
) -> pd.DataFrame:
    """Compute consequence-oriented labels for one project.

    The risk score is a weighted sum of three min-max-normalized components:
    ``n_bugfix_commits_future`` (weight 0.5), ``future_churn`` (weight 0.3)
    and ``n_szz_fixes_future`` (weight 0.2). Files ranked in the top
    ``percentile`` percent within the project are flagged as high-risk.

    Parameters
    ----------
    project_id :
        Which project to compute for.
    snapshot :
        Snapshot date ``t`` for this project.
    commits, changes, szz :
        Cleaned DataFrames from Stage 3.
    jira :
        Optional Jira DataFrame; if supplied an extra
        ``n_jira_bug_commits_future`` column is added for diagnostics
        (not part of the primary score).
    window_months :
        Observation window length.
    percentile :
        Fraction of files labeled positive, in ``[0, 100]``.
    weights :
        Dictionary of component weights. Keys must be a subset of
        ``{"bugfix_commits_future", "future_churn", "szz_defects_future"}``.

    Returns
    -------
    DataFrame with columns:
    ``project_id``, ``basename``, component counts, normalized components
    (``*_norm``), ``risk_score``, ``is_high_risk``.
    """
    universe = basename_universe_at_snapshot(changes, project_id, snapshot)
    if universe.empty:
        return universe.assign(is_high_risk=False)

    bf = bugfix_touches_in_window(commits, changes, project_id, snapshot, window_months)
    ch = churn_in_window(changes, project_id, snapshot, window_months)
    sz = szz_events_in_window(szz, changes, project_id, snapshot, window_months)

    df = universe.merge(bf, on="basename", how="left")
    df = df.merge(ch, on="basename", how="left")
    df = df.merge(sz, on="basename", how="left")

    if jira is not None:
        jb = jira_bug_commits_in_window(commits, jira, changes, project_id, snapshot, window_months)
        df = df.merge(jb, on="basename", how="left")
        df["n_jira_bug_commits_future"] = df["n_jira_bug_commits_future"].fillna(0).astype("int64")

    count_cols = [
        "n_bugfix_commits_future",
        "bugfix_churn_future",
        "future_churn",
        "future_add",
        "future_removed",
        "future_commits",
        "n_szz_fixes_future",
        "n_szz_inducing_past",
    ]
    for c in count_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0).astype("int64")

    df["bugfix_norm"] = _min_max_norm(df["n_bugfix_commits_future"])
    df["churn_norm"] = _min_max_norm(df["future_churn"])
    df["szz_norm"] = _min_max_norm(df["n_szz_fixes_future"])

    w_b = weights.get("bugfix_commits_future", 0.5)
    w_c = weights.get("future_churn", 0.3)
    w_s = weights.get("szz_defects_future", 0.2)
    total_w = w_b + w_c + w_s
    if total_w == 0:
        raise ValueError("RISK_SCORE_WEIGHTS sum to zero")

    df["risk_score"] = (
        w_b * df["bugfix_norm"] + w_c * df["churn_norm"] + w_s * df["szz_norm"]
    ) / total_w

    df["is_high_risk"] = _top_percentile(df["risk_score"], percentile)
    df["is_high_risk"] = df["is_high_risk"].astype(bool)
    return df


# ---------------------------------------------------------------------------
# Severity-based baseline labeling
# ---------------------------------------------------------------------------
_SEVERITY_ORDER = {"INFO": 0, "MINOR": 1, "MAJOR": 2, "CRITICAL": 3, "BLOCKER": 4}


def compute_severity_labels(
    project_id: str,
    snapshot: pd.Timestamp,
    sonar_issues: pd.DataFrame,
    changes: pd.DataFrame,
    high_risk_levels: Iterable[str] = SEVERITY_BASELINE_LEVELS,
) -> pd.DataFrame:
    """Compute SonarQube severity-based labels for one project.

    A basename is labeled high-risk iff it has at least one SonarQube issue
    open at ``snapshot`` whose ``SEVERITY`` is in ``high_risk_levels``
    (default ``("BLOCKER", "CRITICAL")``).

    Returns a DataFrame with columns:
    ``project_id``, ``basename``, ``n_blocker``, ``n_critical``,
    ``n_major``, ``n_minor``, ``n_info``, ``max_severity_rank``,
    ``is_high_risk``.
    """
    universe = basename_universe_at_snapshot(changes, project_id, snapshot)
    if universe.empty:
        return universe.assign(is_high_risk=False)

    open_iss = open_issues_at_snapshot(sonar_issues, project_id, snapshot)
    if open_iss.empty:
        df = universe.copy()
        for col in ("n_blocker", "n_critical", "n_major", "n_minor", "n_info"):
            df[col] = 0
        df["max_severity_rank"] = 0
        df["is_high_risk"] = False
        return df

    sev_counts = (
        open_iss.pivot_table(
            index="basename", columns="SEVERITY", values="ISSUE_KEY", aggfunc="count", fill_value=0
        )
        .rename(
            columns={
                "BLOCKER": "n_blocker",
                "CRITICAL": "n_critical",
                "MAJOR": "n_major",
                "MINOR": "n_minor",
                "INFO": "n_info",
            }
        )
        .reset_index()
    )
    for col in ("n_blocker", "n_critical", "n_major", "n_minor", "n_info"):
        if col not in sev_counts.columns:
            sev_counts[col] = 0

    df = universe.merge(sev_counts, on="basename", how="left")
    for col in ("n_blocker", "n_critical", "n_major", "n_minor", "n_info"):
        df[col] = df[col].fillna(0).astype("int64")

    def _max_rank(row) -> int:
        rank = 0
        if row["n_blocker"] > 0:
            rank = 4
        elif row["n_critical"] > 0:
            rank = 3
        elif row["n_major"] > 0:
            rank = 2
        elif row["n_minor"] > 0:
            rank = 1
        elif row["n_info"] > 0:
            rank = 0
        return rank

    df["max_severity_rank"] = df.apply(_max_rank, axis=1).astype("int64")

    levels = {s.upper() for s in high_risk_levels}
    flags = (
        (("BLOCKER" in levels) & (df["n_blocker"] > 0))
        | (("CRITICAL" in levels) & (df["n_critical"] > 0))
        | (("MAJOR" in levels) & (df["n_major"] > 0))
        | (("MINOR" in levels) & (df["n_minor"] > 0))
        | (("INFO" in levels) & (df["n_info"] > 0))
    )
    df["is_high_risk"] = flags.astype(bool)
    return df


# ---------------------------------------------------------------------------
# SZZ defect-oriented baseline labeling
# ---------------------------------------------------------------------------
def compute_szz_labels(
    project_id: str,
    snapshot: pd.Timestamp,
    changes: pd.DataFrame,
    szz: pd.DataFrame,
    window_months: int = OBSERVATION_WINDOW_MONTHS,
) -> pd.DataFrame:
    """SZZ defect-oriented labels: basename is positive iff a fault-fixing
    commit within the window modified it.

    This is the classical SZZ-based defect-prediction target, adapted to
    basename granularity.
    """
    universe = basename_universe_at_snapshot(changes, project_id, snapshot)
    if universe.empty:
        return universe.assign(is_high_risk=False)

    sz = szz_events_in_window(szz, changes, project_id, snapshot, window_months)
    df = universe.merge(sz, on="basename", how="left")
    for c in ("n_szz_fixes_future", "n_szz_inducing_past"):
        if c in df.columns:
            df[c] = df[c].fillna(0).astype("int64")
        else:
            df[c] = 0
    df["is_high_risk"] = df["n_szz_fixes_future"] > 0
    return df


# ---------------------------------------------------------------------------
# Label agreement
# ---------------------------------------------------------------------------
def label_agreement(labels_by_variant: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Pairwise Cohen's kappa and Jaccard between label variants.

    Each value in ``labels_by_variant`` must be a DataFrame containing
    ``project_id``, ``basename`` and ``is_high_risk`` columns.
    """
    from itertools import combinations

    # Align all variants on the union of (project_id, basename)
    keyed = {
        name: df[["project_id", "basename", "is_high_risk"]]
        .rename(columns={"is_high_risk": name})
        for name, df in labels_by_variant.items()
    }
    merged = None
    for name, df in keyed.items():
        merged = df if merged is None else merged.merge(df, on=["project_id", "basename"], how="outer")
    for name in keyed:
        merged[name] = merged[name].fillna(False).astype(bool)

    rows = []
    names = list(keyed.keys())
    for a, b in combinations(names, 2):
        ya, yb = merged[a].values, merged[b].values
        pa = ya.mean()
        pb = yb.mean()
        p_obs = (ya == yb).mean()
        p_exp = pa * pb + (1 - pa) * (1 - pb)
        kappa = (p_obs - p_exp) / (1 - p_exp) if p_exp < 1 else np.nan
        inter = (ya & yb).sum()
        union = (ya | yb).sum()
        jaccard = inter / union if union > 0 else np.nan
        rows.append(
            {
                "variant_a": a,
                "variant_b": b,
                "positives_a": int(ya.sum()),
                "positives_b": int(yb.sum()),
                "agreement_pct": round(p_obs * 100, 2),
                "cohen_kappa": round(kappa, 4) if np.isfinite(kappa) else np.nan,
                "jaccard": round(jaccard, 4) if np.isfinite(jaccard) else np.nan,
                "intersection": int(inter),
                "union": int(union),
            }
        )
    return pd.DataFrame(rows)


**`src/features/static_features.py`** - Per-basename SonarQube + project-context features at t.

In [ ]:
%%writefile /content/src/features/static_features.py
"""
Snapshot-aware static feature extraction at (project, basename) granularity.

This module consumes the cleaned parquets produced by Stage 3 and emits one
row per (project, basename) with:

- **Per-basename aggregated SonarQube issue features at snapshot ``t``**
  (counts by severity / by type, technical-debt minutes, distinct rules).
  These are the "static code features" of Proposal Table 1 at the
  granularity imposed by the dataset (see RESEARCH_LOG.md 2026-04-23).
- **Project-level context features from the most recent
  ``SONAR_ANALYSIS`` with ``analysis_date <= t``**, replicated for every
  basename in the project. These capture project-wide maintainability
  indicators (NCLOC, COMPLEXITY, SQALE_INDEX, COVERAGE, etc.) as the
  macro-context each file lives in.
- **Size proxy from Git history**: cumulative ``LINES_ADDED`` minus
  ``LINES_REMOVED`` up to ``t`` (floored at zero) - a file-level
  substitute for per-file NCLOC that the dataset does not store.

Columns that could leak the severity-baseline label (``n_blocker``,
``n_critical``, etc.) are computed here but the dataset-build stage
selectively drops them when training on the severity variant.

References
----------
- Lenarduzzi, V., et al. (2019). The Technical Debt Dataset. PROMISE.
- Tsoukalas, D., et al. (2020). Machine learning for technical debt
  identification. IEEE TSE.
"""
from __future__ import annotations

import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from src.data.szz import basename_universe_at_snapshot, open_issues_at_snapshot  # noqa: E402


# ---------------------------------------------------------------------------
# Issue-based static features per basename at snapshot
# ---------------------------------------------------------------------------
def sonar_issue_features_at_snapshot(
    sonar_issues: pd.DataFrame,
    project_id: str,
    snapshot: pd.Timestamp,
) -> pd.DataFrame:
    """Aggregate SonarQube issues open at ``snapshot`` into per-basename features.

    Columns produced (all non-negative integers or floats):
    - ``n_issues_open``
    - ``n_blocker``, ``n_critical``, ``n_major``, ``n_minor``, ``n_info``
    - ``n_code_smell``, ``n_bug``, ``n_vulnerability``
    - ``total_debt_minutes`` - sum of ``DEBT`` (minutes of remediation effort)
    - ``total_effort_minutes`` - sum of ``EFFORT`` (if present)
    - ``n_distinct_rules``
    - ``max_severity_rank`` in ``{0..4}`` where INFO=0 and BLOCKER=4
    """
    open_iss = open_issues_at_snapshot(sonar_issues, project_id, snapshot)
    if open_iss.empty:
        return pd.DataFrame(
            columns=[
                "basename",
                "n_issues_open",
                "n_blocker",
                "n_critical",
                "n_major",
                "n_minor",
                "n_info",
                "n_code_smell",
                "n_bug",
                "n_vulnerability",
                "total_debt_minutes",
                "total_effort_minutes",
                "n_distinct_rules",
                "max_severity_rank",
            ]
        )

    rank_map = {"INFO": 0, "MINOR": 1, "MAJOR": 2, "CRITICAL": 3, "BLOCKER": 4}
    open_iss = open_iss.assign(_rank=open_iss["SEVERITY"].map(rank_map).fillna(0).astype("int64"))

    # Severity counts via pivot
    sev = (
        open_iss.pivot_table(
            index="basename", columns="SEVERITY", values="ISSUE_KEY", aggfunc="count", fill_value=0
        )
        .rename(
            columns={
                "BLOCKER": "n_blocker",
                "CRITICAL": "n_critical",
                "MAJOR": "n_major",
                "MINOR": "n_minor",
                "INFO": "n_info",
            }
        )
    )
    for col in ("n_blocker", "n_critical", "n_major", "n_minor", "n_info"):
        if col not in sev.columns:
            sev[col] = 0

    # Type counts via pivot
    typ = (
        open_iss.pivot_table(
            index="basename", columns="TYPE", values="ISSUE_KEY", aggfunc="count", fill_value=0
        )
        .rename(
            columns={
                "CODE_SMELL": "n_code_smell",
                "BUG": "n_bug",
                "VULNERABILITY": "n_vulnerability",
            }
        )
    )
    for col in ("n_code_smell", "n_bug", "n_vulnerability"):
        if col not in typ.columns:
            typ[col] = 0

    agg = open_iss.groupby("basename").agg(
        n_issues_open=("ISSUE_KEY", "count"),
        total_debt_minutes=("DEBT", "sum"),
        total_effort_minutes=("EFFORT", "sum"),
        n_distinct_rules=("RULE", "nunique"),
        max_severity_rank=("_rank", "max"),
    )

    out = agg.join(sev).join(typ).reset_index()
    for c in (
        "n_blocker",
        "n_critical",
        "n_major",
        "n_minor",
        "n_info",
        "n_code_smell",
        "n_bug",
        "n_vulnerability",
        "n_issues_open",
        "n_distinct_rules",
        "max_severity_rank",
    ):
        out[c] = out[c].fillna(0).astype("int64")
    for c in ("total_debt_minutes", "total_effort_minutes"):
        out[c] = out[c].fillna(0.0).astype(float)
    return out


# ---------------------------------------------------------------------------
# Git-derived pseudo size at snapshot
# ---------------------------------------------------------------------------
def git_pseudo_size_at_snapshot(
    changes: pd.DataFrame,
    project_id: str,
    snapshot: pd.Timestamp,
) -> pd.DataFrame:
    """Cumulative added - removed lines up to snapshot, floored at zero.

    A file-level approximation of current NCLOC when true per-file LOC is not
    available in the dataset. See Proposal 3.3 and thesis Threats to
    Construct Validity.
    """
    t = snapshot
    if t.tz is None:
        t = t.tz_localize("UTC")
    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["DATE"] <= t]
    if ch.empty:
        return pd.DataFrame(columns=["basename", "pseudo_ncloc_at_t"])
    g = ch.groupby("basename").agg(
        _add_sum=("LINES_ADDED", "sum"),
        _rem_sum=("LINES_REMOVED", "sum"),
    )
    g["pseudo_ncloc_at_t"] = (g["_add_sum"] - g["_rem_sum"]).clip(lower=0).astype("int64")
    return g[["pseudo_ncloc_at_t"]].reset_index()


# ---------------------------------------------------------------------------
# Project-level context features at snapshot
# ---------------------------------------------------------------------------
_PROJECT_CONTEXT_COLS: tuple[str, ...] = (
    # Size
    "ncloc",
    "lines",
    "classes",
    "files",
    "functions",
    "statements",
    "comment_lines",
    # Complexity
    "complexity",
    "cognitive_complexity",
    "file_complexity",
    "function_complexity",
    "class_complexity",
    # Density / quality
    "comment_lines_density",
    "duplicated_lines",
    "duplicated_lines_density",
    "duplicated_blocks",
    "duplicated_files",
    "coverage",
    "line_coverage",
    "lines_to_cover",
    "uncovered_lines",
    # Technical debt at project scope (structural context - not label leakage)
    "sqale_index",
    "sqale_debt_ratio",
    "sqale_rating",
    "reliability_rating",
    "security_rating",
    "reliability_remediation_effort",
    "security_remediation_effort",
    "open_issues",
)


def project_context_at_snapshot(
    sonar_measures: pd.DataFrame,
    project_id: str,
    snapshot: pd.Timestamp,
) -> dict:
    """Return project-level SonarQube measures from the latest analysis <= ``t``.

    Returns an empty dict if no analysis exists before the snapshot (very rare,
    would indicate a misconfigured project).
    """
    t = snapshot
    if t.tz is None:
        t = t.tz_localize("UTC")
    m = sonar_measures[sonar_measures["project_id"] == project_id]
    m = m[m["analysis_date"] <= t]
    if m.empty:
        return {}
    # Take the most recent analysis at or before t
    row = m.sort_values("analysis_date").iloc[-1]
    out = {f"project_{c}": row.get(c) for c in _PROJECT_CONTEXT_COLS if c in m.columns}
    out["project_context_analysis_date"] = row["analysis_date"]
    return out


# ---------------------------------------------------------------------------
# Combined static features for a single project
# ---------------------------------------------------------------------------
def build_static_features_for_project(
    project_id: str,
    snapshot: pd.Timestamp,
    sonar_issues: pd.DataFrame,
    sonar_measures: pd.DataFrame,
    changes: pd.DataFrame,
) -> pd.DataFrame:
    """Assemble the full per-basename static feature row for one project."""
    universe = basename_universe_at_snapshot(changes, project_id, snapshot)
    if universe.empty:
        return universe

    issue_feats = sonar_issue_features_at_snapshot(sonar_issues, project_id, snapshot)
    size_feats = git_pseudo_size_at_snapshot(changes, project_id, snapshot)
    ctx = project_context_at_snapshot(sonar_measures, project_id, snapshot)

    df = universe.merge(issue_feats, on="basename", how="left")
    df = df.merge(size_feats, on="basename", how="left")

    # Fill issue-count NaNs (files with zero issues) with zero
    issue_fill_zero = [
        "n_issues_open",
        "n_blocker",
        "n_critical",
        "n_major",
        "n_minor",
        "n_info",
        "n_code_smell",
        "n_bug",
        "n_vulnerability",
        "n_distinct_rules",
        "max_severity_rank",
    ]
    for c in issue_fill_zero:
        if c in df.columns:
            df[c] = df[c].fillna(0).astype("int64")
    for c in ("total_debt_minutes", "total_effort_minutes"):
        if c in df.columns:
            df[c] = df[c].fillna(0.0).astype(float)
    if "pseudo_ncloc_at_t" in df.columns:
        df["pseudo_ncloc_at_t"] = df["pseudo_ncloc_at_t"].fillna(0).astype("int64")

    # Derived ratios
    df["issue_density"] = np.where(
        df.get("pseudo_ncloc_at_t", 0) > 0,
        df["n_issues_open"] / df["pseudo_ncloc_at_t"].replace(0, np.nan),
        0.0,
    ).astype(float)
    df["debt_per_loc"] = np.where(
        df.get("pseudo_ncloc_at_t", 0) > 0,
        df["total_debt_minutes"] / df["pseudo_ncloc_at_t"].replace(0, np.nan),
        0.0,
    ).astype(float)

    # Attach project-level context (same for every row of this project)
    for k, v in ctx.items():
        df[k] = v

    df["snapshot_date"] = snapshot
    return df


**`src/features/historical_features.py`** - Per-basename Git process features (pre-snapshot).

In [ ]:
%%writefile /content/src/features/historical_features.py
"""
Snapshot-aware historical (process) feature extraction at (project,
basename) granularity.

Consumes the cleaned ``GIT_COMMITS`` and ``GIT_COMMITS_CHANGES`` parquets
produced in Stage 3 and restricts every statistic to events with
``AUTHOR_DATE <= snapshot``. This is the temporal honesty guarantee:
**no feature value depends on a commit that happened after the snapshot
used to derive its labels**.

Features produced (``_pre`` suffix denotes pre-snapshot window):

- ``total_commits_pre`` - distinct commits touching the basename.
- ``total_contributors_pre`` - distinct authors.
- ``code_added_pre``, ``code_removed_pre`` - sum of lines.
- ``code_churn_pre`` - added + removed.
- ``recent_churn_30d_pre`` / ``recent_churn_90d_pre`` - churn in the
  last 30 / 90 days before ``t``.
- ``recent_commits_30d_pre`` / ``recent_commits_90d_pre`` - commit
  counts in the last 30 / 90 days before ``t``.
- ``file_age_days_at_snapshot`` - days between first commit and ``t``.
- ``days_since_last_change_at_snapshot`` - days between last pre-``t``
  commit and ``t``.
- ``ownership_ratio_pre`` - max author's commits / total commits.
- ``avg_change_size_pre``, ``max_single_commit_churn_pre``,
  ``std_change_size_pre`` - distribution of per-commit churn.

No feature uses ``CHANGE_TYPE`` (not available in TD Dataset v2.0);
change-type counts are replaced by distributional churn statistics
(max, std) which capture volatility without needing those annotations.
"""
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from src.data.szz import basename_universe_at_snapshot  # noqa: E402


def _ensure_utc(ts: pd.Timestamp) -> pd.Timestamp:
    ts = pd.Timestamp(ts)
    if ts.tz is None:
        ts = ts.tz_localize("UTC")
    return ts


def build_historical_features_for_project(
    project_id: str,
    snapshot: pd.Timestamp,
    commits: pd.DataFrame,
    changes: pd.DataFrame,
) -> pd.DataFrame:
    """Per-basename historical features for one project at snapshot ``t``.

    Parameters
    ----------
    project_id :
        Project key to compute for.
    snapshot :
        Snapshot date ``t``; only commits with ``AUTHOR_DATE <= t`` are used.
    commits :
        ``clean_git_commits`` parquet DataFrame.
    changes :
        ``clean_git_commits_changes`` parquet DataFrame, one row per
        (commit, file) with ``basename``, ``LINES_ADDED``, ``LINES_REMOVED``.
    """
    t = _ensure_utc(snapshot)
    universe = basename_universe_at_snapshot(changes, project_id, t)
    if universe.empty:
        return universe

    # Restrict commits and changes to this project, pre-snapshot only
    c = commits[commits["PROJECT_ID"] == project_id]
    c = c[c["AUTHOR_DATE"] <= t][["COMMIT_HASH", "AUTHOR_DATE", "AUTHOR"]]

    ch = changes[changes["PROJECT_ID"] == project_id]
    ch = ch[ch["DATE"] <= t].copy()

    # Attach author info onto each change row so we can compute author-level aggregates
    ch = ch.merge(c[["COMMIT_HASH", "AUTHOR_DATE", "AUTHOR"]], on="COMMIT_HASH", how="left")
    ch["row_churn"] = ch["LINES_ADDED"].astype("int64") + ch["LINES_REMOVED"].astype("int64")

    t_30 = t - pd.Timedelta(days=30)
    t_90 = t - pd.Timedelta(days=90)
    ch["in_30d"] = (ch["AUTHOR_DATE"] > t_30) & (ch["AUTHOR_DATE"] <= t)
    ch["in_90d"] = (ch["AUTHOR_DATE"] > t_90) & (ch["AUTHOR_DATE"] <= t)

    # ----- Basename-level aggregates over all pre-snapshot commits -----
    base = ch.groupby("basename").agg(
        total_commits_pre=("COMMIT_HASH", "nunique"),
        total_contributors_pre=("AUTHOR", "nunique"),
        code_added_pre=("LINES_ADDED", "sum"),
        code_removed_pre=("LINES_REMOVED", "sum"),
        first_commit_date=("AUTHOR_DATE", "min"),
        last_commit_date=("AUTHOR_DATE", "max"),
        avg_change_size_pre=("row_churn", "mean"),
        max_single_commit_churn_pre=("row_churn", "max"),
        std_change_size_pre=("row_churn", "std"),
    )
    base["code_churn_pre"] = (base["code_added_pre"] + base["code_removed_pre"]).astype("int64")

    # ----- Recency windows -----
    def _window_agg(col_mask: str, out_churn: str, out_commits: str) -> pd.DataFrame:
        sub = ch[ch[col_mask]]
        if sub.empty:
            return pd.DataFrame(columns=["basename", out_churn, out_commits]).set_index("basename")
        return (
            sub.groupby("basename")
            .agg(**{out_churn: ("row_churn", "sum"), out_commits: ("COMMIT_HASH", "nunique")})
        )

    w30 = _window_agg("in_30d", "recent_churn_30d_pre", "recent_commits_30d_pre")
    w90 = _window_agg("in_90d", "recent_churn_90d_pre", "recent_commits_90d_pre")

    # ----- Ownership: max author commits / total commits -----
    author_commits = (
        ch.groupby(["basename", "AUTHOR"])["COMMIT_HASH"]
        .nunique()
        .reset_index(name="author_commits")
    )
    top_author = author_commits.groupby("basename")["author_commits"].max().rename("max_commits_by_author")

    # ----- Join everything -----
    df = universe.merge(base.reset_index(), on="basename", how="left")
    df = df.merge(w30.reset_index(), on="basename", how="left")
    df = df.merge(w90.reset_index(), on="basename", how="left")
    df = df.merge(top_author.reset_index(), on="basename", how="left")

    # ----- Derived fields -----
    df["file_age_days_at_snapshot"] = (t - df["first_commit_date"]).dt.days
    df["days_since_last_change_at_snapshot"] = (t - df["last_commit_date"]).dt.days
    df["ownership_ratio_pre"] = np.where(
        df["total_commits_pre"] > 0,
        df["max_commits_by_author"] / df["total_commits_pre"],
        np.nan,
    )

    # Fill NaNs: numeric aggregates default to 0, std defaults to 0, dates stay NaT
    zero_fill_cols = [
        "total_commits_pre",
        "total_contributors_pre",
        "code_added_pre",
        "code_removed_pre",
        "code_churn_pre",
        "recent_churn_30d_pre",
        "recent_commits_30d_pre",
        "recent_churn_90d_pre",
        "recent_commits_90d_pre",
        "max_single_commit_churn_pre",
    ]
    for c_ in zero_fill_cols:
        if c_ in df.columns:
            df[c_] = df[c_].fillna(0).astype("int64")

    for c_ in ("avg_change_size_pre", "std_change_size_pre", "ownership_ratio_pre"):
        if c_ in df.columns:
            df[c_] = df[c_].fillna(0.0).astype(float)

    for c_ in ("file_age_days_at_snapshot", "days_since_last_change_at_snapshot"):
        if c_ in df.columns:
            df[c_] = df[c_].fillna(0).astype("int64")

    df["snapshot_date"] = t
    return df.drop(columns=["first_commit_date", "last_commit_date", "max_commits_by_author"])


**`src/models/train.py`** - Within-project stratified K-fold + metric battery.

In [ ]:
%%writefile /content/src/models/train.py
"""
Within-project stratified K-fold training and evaluation.

For a single ``(variant, model)`` pair this module:

1. Loads the corresponding ``dataset_{variant}.parquet`` from Stage 6.
2. Separates numeric feature columns from metadata and labels (dropping
   ``risk_score`` for the consequence variant so that the score that
   defined the label is not used as an input feature).
3. Runs stratified K-fold cross-validation (``config.CV_FOLDS``), with
   optional class-imbalance handling via class-weights (default) or
   SMOTE oversampling (``use_smote=True``).
4. Computes a full metric battery per fold: precision, recall, F1,
   ROC-AUC, PR-AUC (average precision), MCC, and Cost-Effectiveness at
   top-20 percent (``CE@20``).

The driver script ``scripts/07_train.py`` iterates over all variants and
models and writes a tidy results table.

References
----------
- Kamei et al. (2013). Just-in-time quality assurance. IEEE TSE.
- Menzies et al. (2007). Data mining static code attributes. IEEE TSE.
- Saito and Rehmsmeier (2015). PR vs ROC evaluation for imbalanced data.
  PLOS ONE 10(3).
"""
from __future__ import annotations

import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any, Iterable, Optional

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import (  # noqa: E402
    COST_EFFECTIVENESS_AT,
    CV_FOLDS,
    PROCESSED_DATA_DIR,
    RANDOM_STATE,
)


KEY_COLS = ("project_id", "basename")
LABEL_COL = "is_high_risk"
DROP_FOR_CONSEQUENCE = ("risk_score",)


# ---------------------------------------------------------------------------
# Model zoo
# ---------------------------------------------------------------------------
def _make_model(name: str):
    """Build an sklearn estimator by name, using sane defaults from config.

    XGBoost / LightGBM are optional imports: if not installed, those
    models are silently skipped.
    """
    name = name.lower()
    if name == "decision_tree":
        return DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced")
    if name == "random_forest":
        return RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
        )
    if name == "logistic_regression":
        return Pipeline(
            [
                ("scale", StandardScaler(with_mean=False)),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=5000,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                        solver="lbfgs",
                    ),
                ),
            ]
        )
    if name == "svm":
        return Pipeline(
            [
                ("scale", StandardScaler(with_mean=False)),
                (
                    "clf",
                    SVC(
                        kernel="rbf",
                        probability=True,
                        class_weight="balanced",
                        random_state=RANDOM_STATE,
                    ),
                ),
            ]
        )
    if name == "xgboost":
        try:
            from xgboost import XGBClassifier
        except ImportError:
            return None
        return XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            eval_metric="logloss",
            n_jobs=-1,
        )
    if name == "lightgbm":
        try:
            from lightgbm import LGBMClassifier
        except ImportError:
            return None
        return LGBMClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            class_weight="balanced",
            n_jobs=-1,
            verbose=-1,
        )
    raise ValueError(f"Unknown model: {name}")


# ---------------------------------------------------------------------------
# Metric battery
# ---------------------------------------------------------------------------
def _cost_effectiveness_at_k(y_true: np.ndarray, y_score: np.ndarray, k: float) -> float:
    """Fraction of positives captured by the top-``k``% predicted-risk files.

    A prioritization-oriented metric: if a tool inspects only the top
    ``k``% highest-scoring files, CE@k is the recall achieved in that
    budget - used e.g. in Menzies et al. 2007.
    """
    n = len(y_true)
    total_pos = int(y_true.sum())
    if n == 0 or total_pos == 0:
        return 0.0
    budget = max(1, int(np.ceil(n * k)))
    top_idx = np.argsort(-y_score)[:budget]
    return float(y_true[top_idx].sum() / total_pos)


def _metric_row(y_true, y_pred, y_score) -> dict[str, float]:
    return {
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_true, y_score)) if len(set(y_true)) > 1 else float("nan"),
        "pr_auc": float(average_precision_score(y_true, y_score)) if len(set(y_true)) > 1 else float("nan"),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
        "ce_at_20": _cost_effectiveness_at_k(y_true, y_score, COST_EFFECTIVENESS_AT),
    }


# ---------------------------------------------------------------------------
# Data prep
# ---------------------------------------------------------------------------
def load_variant_matrix(variant: str) -> tuple[pd.DataFrame, pd.Series, pd.Series]:
    """Return ``(X, y, project_id)`` for the specified label variant.

    - ``X`` - numeric feature matrix only (non-numeric columns dropped).
    - ``y`` - ``is_high_risk`` as int.
    - ``project_id`` - group column for group-aware validation.
    """
    df = pd.read_parquet(PROCESSED_DATA_DIR / f"dataset_{variant}.parquet")
    drop_cols = list(KEY_COLS) + [LABEL_COL]
    if variant == "consequence":
        for c in DROP_FOR_CONSEQUENCE:
            if c in df.columns:
                drop_cols.append(c)
    X = df.drop(columns=drop_cols, errors="ignore")
    X = X.select_dtypes(include="number")
    y = df[LABEL_COL].astype(int)
    proj = df["project_id"]
    return X, y, proj


# ---------------------------------------------------------------------------
# K-fold driver
# ---------------------------------------------------------------------------
@dataclass
class FoldResult:
    variant: str
    model: str
    fold: int
    n_train: int
    n_test: int
    n_pos_test: int
    metrics: dict[str, float] = field(default_factory=dict)


def stratified_kfold_cv(
    variant: str,
    model_name: str,
    X: pd.DataFrame,
    y: pd.Series,
    n_splits: int = CV_FOLDS,
) -> list[FoldResult]:
    """Run stratified K-fold for one ``(variant, model)`` pair.

    Returns a list of per-fold results containing the full metric
    battery.
    """
    est = _make_model(model_name)
    if est is None:
        return []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    out: list[FoldResult] = []
    yv = y.values
    for fold, (tr, te) in enumerate(skf.split(X, yv), start=1):
        est_f = _make_model(model_name)
        X_tr = X.iloc[tr]
        X_te = X.iloc[te]
        est_f.fit(X_tr, yv[tr])
        if hasattr(est_f, "predict_proba"):
            proba = est_f.predict_proba(X_te)[:, 1]
        else:
            proba = est_f.decision_function(X_te)
        pred = (proba >= 0.5).astype(int)
        metrics = _metric_row(yv[te], pred, proba)
        out.append(
            FoldResult(
                variant=variant,
                model=model_name,
                fold=fold,
                n_train=len(tr),
                n_test=len(te),
                n_pos_test=int(yv[te].sum()),
                metrics=metrics,
            )
        )
    return out


def fold_results_to_frame(results: Iterable[FoldResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        row = {
            "variant": r.variant,
            "model": r.model,
            "fold": r.fold,
            "n_train": r.n_train,
            "n_test": r.n_test,
            "n_pos_test": r.n_pos_test,
            **r.metrics,
        }
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(fold_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate per-fold results into mean +/- std per (variant, model)."""
    metric_cols = [c for c in fold_df.columns if c in {"precision", "recall", "f1", "roc_auc", "pr_auc", "mcc", "ce_at_20"}]
    grp = fold_df.groupby(["variant", "model"])[metric_cols]
    means = grp.mean().add_suffix("_mean")
    stds = grp.std().add_suffix("_std")
    return pd.concat([means, stds], axis=1).reset_index()


**`src/models/cross_project.py`** - Leave-One-Project-Out cross-project validation.

In [ ]:
%%writefile /content/src/models/cross_project.py
"""
Leave-One-Project-Out (LOPO) cross-project validation.

For each label variant and each model, this module runs N = 22 folds
where the test set is one held-out project and the training set is the
union of the other 21 projects. This is the strongest generalization
test for software-engineering ML: it answers "if I train on 21 projects
and deploy on a brand-new project, what performance should I expect?"

Public API
----------
- ``lopo_cv(variant, model_name)`` - return a DataFrame with one row
  per held-out project containing the full metric battery.
- ``lopo_summary(folds_df)`` - aggregate per-variant, per-model means
  and standard deviations across held-out projects.

Metrics are identical to the within-project module so results are
directly comparable (difference = generalization gap).

References
----------
- Zimmermann et al. (2009). Cross-project defect prediction. FSE.
- Herbold et al. (2018). A comparative study to benchmark cross-project
  defect prediction approaches. IEEE TSE 44(9).
"""
from __future__ import annotations

import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import (  # noqa: E402
    COST_EFFECTIVENESS_AT,
    PROCESSED_DATA_DIR,
)
from src.models.train import (  # noqa: E402
    KEY_COLS,
    LABEL_COL,
    _cost_effectiveness_at_k,
    _make_model,
    _metric_row,
    load_variant_matrix,
)


@dataclass
class LopoFoldResult:
    variant: str
    model: str
    held_out_project: str
    n_train: int
    n_test: int
    n_pos_test: int
    metrics: dict[str, float] = field(default_factory=dict)


def lopo_cv(
    variant: str,
    model_name: str,
) -> list[LopoFoldResult]:
    """Run LOPO CV for one ``(variant, model)`` pair.

    Returns one result per held-out project. Projects where the
    training set contains no positive labels are skipped (degenerate).
    """
    X, y, proj = load_variant_matrix(variant)
    projects = sorted(proj.unique())
    out: list[LopoFoldResult] = []

    for held_out in projects:
        te_mask = (proj == held_out).values
        tr_mask = ~te_mask

        y_tr = y.values[tr_mask]
        y_te = y.values[te_mask]
        # Require both classes in training and at least one positive in test
        if len(set(y_tr)) < 2 or y_te.sum() == 0:
            continue

        est = _make_model(model_name)
        if est is None:
            return []

        X_tr = X.iloc[tr_mask]
        X_te = X.iloc[te_mask]
        est.fit(X_tr, y_tr)
        if hasattr(est, "predict_proba"):
            proba = est.predict_proba(X_te)[:, 1]
        else:
            proba = est.decision_function(X_te)
        pred = (proba >= 0.5).astype(int)
        metrics = _metric_row(y_te, pred, proba)

        out.append(
            LopoFoldResult(
                variant=variant,
                model=model_name,
                held_out_project=held_out,
                n_train=int(tr_mask.sum()),
                n_test=int(te_mask.sum()),
                n_pos_test=int(y_te.sum()),
                metrics=metrics,
            )
        )
    return out


def lopo_results_to_frame(results: Iterable[LopoFoldResult]) -> pd.DataFrame:
    rows = []
    for r in results:
        rows.append(
            {
                "variant": r.variant,
                "model": r.model,
                "held_out_project": r.held_out_project,
                "n_train": r.n_train,
                "n_test": r.n_test,
                "n_pos_test": r.n_pos_test,
                **r.metrics,
            }
        )
    return pd.DataFrame(rows)


def lopo_summary(fold_df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate LOPO per-project metrics into mean +/- std per ``(variant, model)``."""
    metric_cols = [
        c for c in fold_df.columns
        if c in {"precision", "recall", "f1", "roc_auc", "pr_auc", "mcc", "ce_at_20"}
    ]
    grp = fold_df.groupby(["variant", "model"])[metric_cols]
    means = grp.mean().add_suffix("_mean")
    stds = grp.std().add_suffix("_std")
    n = grp.count().iloc[:, :1].rename(columns={metric_cols[0]: "n_projects"})
    return pd.concat([n, means, stds], axis=1).reset_index()


**`src/analysis/sensitivity.py`** - Sensitivity grid for consequence labelling.

In [ ]:
%%writefile /content/src/analysis/sensitivity.py
"""
Sensitivity analysis for the consequence-oriented labeling parameters.

Sweeps a grid of ``(observation_window_months, high_risk_percentile)``
and, for each combination, re-derives consequence labels, re-merges
them with the (fixed) feature matrix and runs a standard 10-fold
stratified CV with the best-performing model family (LightGBM).

The resulting table answers the question: **"Are our headline
predictive-power numbers driven by the specific choice of 6-month
window + top-20-percent threshold, or do they hold across a range of
reasonable parameter settings?"** A robust methodology should show
stable or gracefully-degrading metrics across the grid.

Notes
-----
- Features are snapshot-aware and do NOT depend on the observation
  window, so they are computed once.
- Every label rebuild goes through the same ``compute_consequence_labels``
  function as Stage 4, guaranteeing consistent semantics.
- LightGBM is chosen because it won the LOPO comparison in Stage 8;
  using a single strong model keeps this grid tractable
  (9 configurations x 10 folds = 90 fits, approximately 2 minutes).
"""
from __future__ import annotations

import sys
import time
from pathlib import Path
from typing import Iterable

import numpy as np
import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import PROCESSED_DATA_DIR, TABLES_DIR  # noqa: E402
from src.data.labeling import compute_consequence_labels  # noqa: E402
from src.data.snapshot import load_snapshots  # noqa: E402
from src.models.train import (  # noqa: E402
    KEY_COLS,
    fold_results_to_frame,
    stratified_kfold_cv,
)


METRIC_COLS = ["precision", "recall", "f1", "roc_auc", "pr_auc", "mcc", "ce_at_20"]


def _load_raw_for_labeling() -> dict[str, pd.DataFrame]:
    commits = pd.read_parquet(PROCESSED_DATA_DIR / "clean_git_commits.parquet")
    changes = pd.read_parquet(PROCESSED_DATA_DIR / "clean_git_commits_changes.parquet")
    szz = pd.read_parquet(PROCESSED_DATA_DIR / "clean_szz.parquet")
    jira = pd.read_parquet(PROCESSED_DATA_DIR / "clean_jira_issues.parquet")
    for df, cols in (
        (commits, ["AUTHOR_DATE", "COMMITTER_DATE"]),
        (changes, ["DATE"]),
        (szz, ["fix_date", "induce_date"]),
        (jira, ["CREATION_DATE", "RESOLUTION_DATE", "UPDATE_DATE", "COMMIT_DATE"]),
    ):
        for col in cols:
            if col in df.columns and df[col].dtype.kind == "M" and df[col].dt.tz is None:
                df[col] = df[col].dt.tz_localize("UTC")
    return {"commits": commits, "changes": changes, "szz": szz, "jira": jira}


def _recompute_consequence_labels(
    window_months: int,
    percentile: float,
    snapshots: pd.DataFrame,
    raw: dict[str, pd.DataFrame],
) -> pd.DataFrame:
    parts = []
    for _, row in snapshots[snapshots["eligible"]].sort_values("project_id").iterrows():
        df = compute_consequence_labels(
            project_id=row["project_id"],
            snapshot=row["snapshot_date"],
            commits=raw["commits"],
            changes=raw["changes"],
            szz=raw["szz"],
            jira=raw["jira"],
            window_months=window_months,
            percentile=percentile,
        )
        parts.append(df[["project_id", "basename", "is_high_risk"]])
    return pd.concat(parts, ignore_index=True)


def run_sensitivity_grid(
    windows: Iterable[int] = (3, 6, 12),
    percentiles: Iterable[float] = (10.0, 20.0, 30.0),
    model_name: str = "lightgbm",
) -> pd.DataFrame:
    """Run the sensitivity grid and return aggregated metrics per cell."""
    raw = _load_raw_for_labeling()
    snapshots = load_snapshots(PROCESSED_DATA_DIR / "project_snapshots.parquet")
    base = pd.read_parquet(PROCESSED_DATA_DIR / "dataset_consequence.parquet")
    feature_cols = [
        c for c in base.columns
        if c not in set(KEY_COLS) | {"is_high_risk", "risk_score"}
    ]
    features_only = base[list(KEY_COLS) + feature_cols].copy()
    features_numeric = features_only.select_dtypes(include="number").copy()
    features_numeric[list(KEY_COLS)] = features_only[list(KEY_COLS)]

    rows = []
    for w in windows:
        for p in percentiles:
            t0 = time.time()
            labels = _recompute_consequence_labels(w, p, snapshots, raw)
            merged = features_only.merge(labels, on=list(KEY_COLS), how="inner")
            X = merged.drop(columns=list(KEY_COLS) + ["is_high_risk"]).select_dtypes(include="number")
            y = merged["is_high_risk"].astype(int)
            fold_results = stratified_kfold_cv("consequence", model_name, X, y, n_splits=10)
            df = fold_results_to_frame(fold_results)
            means = df[METRIC_COLS].mean().to_dict()
            stds = df[METRIC_COLS].std().to_dict()
            n_pos = int(y.sum())
            rows.append(
                {
                    "window_months": w,
                    "percentile": p,
                    "positives": n_pos,
                    "positive_rate_pct": round(100.0 * y.mean(), 2),
                    **{f"{k}_mean": round(v, 4) for k, v in means.items()},
                    **{f"{k}_std": round(v, 4) for k, v in stds.items()},
                    "elapsed_s": round(time.time() - t0, 2),
                }
            )
    return pd.DataFrame(rows)


**`src/analysis/ablation.py`** - Feature-group ablation.

In [ ]:
%%writefile /content/src/analysis/ablation.py
"""
Feature-group ablation analysis.

Quantifies how much predictive power each family of features contributes
to each label variant. Three training regimes are compared for every
(variant, group) combination:

- **Only this group** - isolates the intrinsic predictive power of the
  group.
- **All features except this group** - reveals the unique marginal
  contribution (a drop from the full-model baseline).
- **All features** (baseline) - printed once per variant for reference.

The groups are defined semantically rather than by column prefix so that
new features added later still map cleanly to a group.

References
----------
- Zimmermann et al. (2007). Predicting defects for Eclipse. PROMISE.
- Rahman, D'Souza, and Devanbu (2013). Sample size vs. bias. MSR.
"""
from __future__ import annotations

import sys
import time
from pathlib import Path
from typing import Iterable

import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from src.models.train import (  # noqa: E402
    KEY_COLS,
    LABEL_COL,
    DROP_FOR_CONSEQUENCE,
    fold_results_to_frame,
    load_variant_matrix,
    stratified_kfold_cv,
)


METRIC_COLS = ["precision", "recall", "f1", "roc_auc", "pr_auc", "mcc", "ce_at_20"]


# Feature-group definitions. Names are matched against column names in
# ``dataset_{variant}.parquet``. Any feature not matched is grouped as
# ``other`` so the ablation is exhaustive.
GROUPS: dict[str, list[str]] = {
    "static_sonar": [
        "n_issues_open",
        "n_blocker",
        "n_critical",
        "n_major",
        "n_minor",
        "n_info",
        "n_code_smell",
        "n_bug",
        "n_vulnerability",
        "total_debt_minutes",
        "total_effort_minutes",
        "n_distinct_rules",
        "max_severity_rank",
        "issue_density",
        "debt_per_loc",
        "pseudo_ncloc_at_t",
    ],
    "historical": [
        "total_commits_pre",
        "total_contributors_pre",
        "code_added_pre",
        "code_removed_pre",
        "code_churn_pre",
        "avg_change_size_pre",
        "max_single_commit_churn_pre",
        "std_change_size_pre",
        "recent_churn_30d_pre",
        "recent_churn_90d_pre",
        "recent_commits_30d_pre",
        "recent_commits_90d_pre",
        "file_age_days_at_snapshot",
        "days_since_last_change_at_snapshot",
        "ownership_ratio_pre",
    ],
    "project_context": [
        # Populated dynamically below - every column starting with "project_"
    ],
}


def _resolve_groups(available_cols: Iterable[str]) -> dict[str, list[str]]:
    """Expand the ``project_context`` placeholder and drop missing cols."""
    avail = set(available_cols)
    resolved = {
        name: sorted(c for c in cols if c in avail)
        for name, cols in GROUPS.items()
        if name != "project_context"
    }
    resolved["project_context"] = sorted(
        c for c in avail if c.startswith("project_") or c == "has_project_context"
    )
    assigned = {c for cols in resolved.values() for c in cols}
    other = sorted(c for c in avail if c not in assigned)
    if other:
        resolved["other"] = other
    return resolved


def _evaluate(X: pd.DataFrame, y: pd.Series, variant: str, model_name: str) -> dict[str, float]:
    fold_results = stratified_kfold_cv(variant, model_name, X, y, n_splits=10)
    df = fold_results_to_frame(fold_results)
    return df[METRIC_COLS].mean().to_dict()


def run_ablation(variant: str, model_name: str = "lightgbm") -> pd.DataFrame:
    """Run the group ablation for one variant and return a tidy table."""
    X, y, _ = load_variant_matrix(variant)
    groups = _resolve_groups(X.columns)
    rows = []

    t0 = time.time()
    base_metrics = _evaluate(X, y, variant, model_name)
    rows.append(
        {
            "variant": variant,
            "group": "all",
            "mode": "all_features",
            "n_features": X.shape[1],
            **{f"{k}_mean": round(v, 4) for k, v in base_metrics.items()},
            "elapsed_s": round(time.time() - t0, 2),
        }
    )

    for name, cols in groups.items():
        if not cols:
            continue

        t1 = time.time()
        metrics_only = _evaluate(X[cols], y, variant, model_name)
        rows.append(
            {
                "variant": variant,
                "group": name,
                "mode": "only_this_group",
                "n_features": len(cols),
                **{f"{k}_mean": round(v, 4) for k, v in metrics_only.items()},
                "elapsed_s": round(time.time() - t1, 2),
            }
        )

        t1 = time.time()
        remaining = [c for c in X.columns if c not in cols]
        if remaining:
            metrics_wo = _evaluate(X[remaining], y, variant, model_name)
            rows.append(
                {
                    "variant": variant,
                    "group": name,
                    "mode": "leave_out_this_group",
                    "n_features": len(remaining),
                    **{f"{k}_mean": round(v, 4) for k, v in metrics_wo.items()},
                    "elapsed_s": round(time.time() - t1, 2),
                }
            )
    return pd.DataFrame(rows)


**`src/analysis/importance.py`** - SHAP + permutation importance.

In [ ]:
%%writefile /content/src/analysis/importance.py
"""
Feature-importance analysis via SHAP and permutation importance.

For each label variant, trains a single LightGBM model on the whole
dataset (stratified 80/20 split for hold-out importance scoring) and
computes:

- **TreeSHAP values** - accurate Shapley values for tree ensembles, used
  to produce the SHAP summary plot and per-feature mean absolute SHAP
  value.
- **Permutation importance** - model-agnostic drop in ROC-AUC when the
  column is shuffled; provides a validation metric against SHAP.

Both rankings are saved as CSV tables; the raw SHAP matrix is persisted
for use by the plotting module.
"""
from __future__ import annotations

import sys
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import RANDOM_STATE  # noqa: E402
from src.models.train import _make_model, load_variant_matrix  # noqa: E402


def compute_importance(
    variant: str,
    model_name: str = "lightgbm",
    n_permutation_repeats: int = 5,
    shap_background_size: int = 1000,
) -> dict[str, pd.DataFrame]:
    """Compute SHAP + permutation importance for one variant.

    Returns a dict with:
    - ``shap_summary`` - DataFrame indexed by feature with mean absolute
      SHAP value, mean SHAP value and signed contribution sign.
    - ``permutation`` - DataFrame with mean and std permutation
      importance per feature (drop in ROC-AUC).
    - ``shap_matrix`` - raw SHAP values (numpy array) and corresponding
      X_sample DataFrame for plotting.
    """
    X, y, proj = load_variant_matrix(variant)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )

    est = _make_model(model_name)
    est.fit(X_train, y_train)

    # ---- SHAP ----
    import shap

    # Use a background sample for expected value computation to bound runtime
    bg_size = min(shap_background_size, len(X_train))
    bg = X_train.sample(n=bg_size, random_state=RANDOM_STATE)
    explainer = shap.TreeExplainer(est, data=bg, feature_perturbation="interventional")
    # Limit to the test split for plotting (faster, consistent with other stats)
    shap_values = explainer.shap_values(X_test, check_additivity=False)
    if isinstance(shap_values, list):
        # binary classifier with per-class SHAP - take class-1 contribution
        shap_values = shap_values[1]

    shap_values = np.asarray(shap_values)

    # Legacy TreeExplainer can return (n_samples, n_features, 2) for binary
    if shap_values.ndim == 3:
        shap_values = shap_values[..., 1]

    shap_abs = np.abs(shap_values).mean(axis=0)
    shap_mean = shap_values.mean(axis=0)
    shap_summary = (
        pd.DataFrame(
            {
                "feature": X_test.columns,
                "mean_abs_shap": shap_abs,
                "mean_shap": shap_mean,
                "sign": np.where(shap_mean >= 0, "+", "-"),
            }
        )
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )

    # ---- Permutation importance ----
    pi = permutation_importance(
        est, X_test, y_test,
        n_repeats=n_permutation_repeats,
        random_state=RANDOM_STATE,
        scoring="roc_auc",
        n_jobs=1,
    )
    perm = (
        pd.DataFrame(
            {
                "feature": X_test.columns,
                "perm_importance_mean": pi.importances_mean,
                "perm_importance_std": pi.importances_std,
            }
        )
        .sort_values("perm_importance_mean", ascending=False)
        .reset_index(drop=True)
    )

    return {
        "shap_summary": shap_summary,
        "permutation": perm,
        "shap_values": shap_values,
        "X_sample": X_test.reset_index(drop=True),
    }


**`src/reporting/figures.py`** - Publication-ready figures (PNG + PDF).

In [ ]:
%%writefile /content/src/reporting/figures.py
"""
Publication-ready figures for the TD prediction thesis.

Every public function writes a 300-DPI PNG and a PDF to ``FIGURES_DIR``
using a consistent greyscale-friendly palette. Figures are designed to
be included directly in the thesis; no further editing is required.

Figures generated:

- ``fig_label_agreement_venn.(png|pdf)`` - 3-way Venn of high-risk sets
- ``fig_per_project_positive_rates.(png|pdf)`` - bar chart per project
- ``fig_within_vs_lopo.(png|pdf)`` - dot-plot of generalization gap
- ``fig_sensitivity_heatmap.(png|pdf)`` - sensitivity ROC / CE@20 grid
- ``fig_feature_ablation.(png|pdf)`` - grouped bar chart
- ``fig_shap_{variant}.(png|pdf)`` - per-variant SHAP summary
- ``fig_lopo_per_project.(png|pdf)`` - LOPO F1 distribution by project
"""
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import FIGURES_DIR, PROCESSED_DATA_DIR, TABLES_DIR  # noqa: E402


PALETTE = {
    "consequence": "#1f77b4",
    "severity": "#d62728",
    "szz": "#2ca02c",
    "all": "#7f7f7f",
    "static_sonar": "#1f77b4",
    "historical": "#ff7f0e",
    "project_context": "#2ca02c",
    "within": "#1f77b4",
    "lopo": "#d62728",
}


def _save(fig: plt.Figure, name: str) -> None:
    fig.tight_layout()
    fig.savefig(FIGURES_DIR / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGURES_DIR / f"{name}.pdf", bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
# 1. Label agreement Venn
# ---------------------------------------------------------------------------
def fig_label_agreement_venn() -> None:
    from matplotlib_venn import venn3

    cons = pd.read_parquet(PROCESSED_DATA_DIR / "labels_consequence.parquet")
    sev = pd.read_parquet(PROCESSED_DATA_DIR / "labels_severity.parquet")
    szz = pd.read_parquet(PROCESSED_DATA_DIR / "labels_szz.parquet")

    def _key(df):
        return set(
            zip(
                df.loc[df["is_high_risk"], "project_id"],
                df.loc[df["is_high_risk"], "basename"],
            )
        )

    s_cons, s_sev, s_szz = _key(cons), _key(sev), _key(szz)

    fig, ax = plt.subplots(figsize=(6, 5))
    v = venn3(
        [s_cons, s_sev, s_szz],
        set_labels=("Consequence\n(top 20%)", "Severity\n(BLOCKER/CRIT)", "SZZ\n(fault-fix in window)"),
        set_colors=(PALETTE["consequence"], PALETTE["severity"], PALETTE["szz"]),
        alpha=0.55,
        ax=ax,
    )
    if v is not None:
        for label in v.set_labels or []:
            if label:
                label.set_fontsize(10)
        for label in v.subset_labels or []:
            if label:
                label.set_fontsize(9)
    ax.set_title("High-Risk Technical Debt: Three-Label Agreement\n(22 projects, 23,911 files)", fontsize=11)
    _save(fig, "fig_label_agreement_venn")


# ---------------------------------------------------------------------------
# 2. Per-project positive rates
# ---------------------------------------------------------------------------
def fig_per_project_positive_rates() -> None:
    df = pd.read_csv(TABLES_DIR / "label_summary.csv")
    df = df.sort_values("n_basenames", ascending=False)

    x = np.arange(len(df))
    w = 0.28

    fig, ax = plt.subplots(figsize=(11, 5))
    ax.bar(x - w, df["consequence_rate_pct"], w, label="Consequence", color=PALETTE["consequence"])
    ax.bar(x, df["severity_rate_pct"], w, label="Severity", color=PALETTE["severity"])
    ax.bar(x + w, df["szz_rate_pct"], w, label="SZZ", color=PALETTE["szz"])

    short = [p.replace("org.apache:", "") for p in df["project_id"]]
    ax.set_xticks(x)
    ax.set_xticklabels(short, rotation=45, ha="right", fontsize=9)
    ax.set_ylabel("Positive rate (%)")
    ax.set_title("Per-project high-risk positive rates by label variant")
    ax.legend(loc="upper right")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    _save(fig, "fig_per_project_positive_rates")


# ---------------------------------------------------------------------------
# 3. Within vs LOPO (dot-plot)
# ---------------------------------------------------------------------------
def fig_within_vs_lopo() -> None:
    gap = pd.read_csv(TABLES_DIR / "lopo_vs_within.csv")
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

    for ax, metric, title in [
        (axes[0], "f1", "F1 score"),
        (axes[1], "ce_at_20", "CE @ top 20%"),
    ]:
        d = gap.dropna(subset=[f"{metric}_within", f"{metric}_lopo"]).copy()
        d["label"] = d["variant"] + " / " + d["model"]
        d = d.sort_values(f"{metric}_lopo")
        y = np.arange(len(d))
        ax.hlines(y, d[f"{metric}_lopo"], d[f"{metric}_within"], color="#888", linewidth=1.2, alpha=0.6)
        ax.scatter(d[f"{metric}_within"], y, color=PALETTE["within"], s=38, label="Within-project", zorder=3)
        ax.scatter(d[f"{metric}_lopo"], y, color=PALETTE["lopo"], s=38, label="LOPO", zorder=3)
        ax.set_yticks(y)
        ax.set_yticklabels(d["label"], fontsize=8)
        ax.set_xlim(0, 1)
        ax.set_xlabel(title)
        ax.grid(axis="x", linestyle=":", alpha=0.5)
        ax.set_title(f"Generalization gap: {title}")
        ax.legend(loc="lower right", fontsize=9)

    fig.suptitle("Within-project CV vs Leave-One-Project-Out", y=1.02, fontsize=12)
    _save(fig, "fig_within_vs_lopo")


# ---------------------------------------------------------------------------
# 4. Sensitivity heatmap
# ---------------------------------------------------------------------------
def fig_sensitivity_heatmap() -> None:
    df = pd.read_csv(TABLES_DIR / "sensitivity_consequence.csv")
    pivots = {
        "roc_auc_mean": "ROC-AUC",
        "f1_mean": "F1",
        "ce_at_20_mean": "CE @ top 20%",
    }
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
    for ax, (col, title) in zip(axes, pivots.items()):
        piv = df.pivot(index="window_months", columns="percentile", values=col)
        sns.heatmap(
            piv,
            annot=True,
            fmt=".3f",
            cmap="viridis",
            ax=ax,
            cbar_kws={"shrink": 0.8},
            linewidths=0.5,
            linecolor="white",
        )
        ax.set_title(title)
        ax.set_xlabel("Percentile threshold (%)")
        ax.set_ylabel("Observation window (months)")

    fig.suptitle("Consequence-variant sensitivity (LightGBM, 10-fold CV)", y=1.04, fontsize=12)
    _save(fig, "fig_sensitivity_heatmap")


# ---------------------------------------------------------------------------
# 5. Feature ablation
# ---------------------------------------------------------------------------
def fig_feature_ablation() -> None:
    df = pd.read_csv(TABLES_DIR / "feature_ablation.csv")
    only = df[df["mode"] == "only_this_group"].copy()

    groups = ["static_sonar", "historical", "project_context"]
    variants = ["consequence", "severity", "szz"]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=False)

    for ax, metric, title in [(axes[0], "f1_mean", "F1"), (axes[1], "ce_at_20_mean", "CE @ top 20%")]:
        x = np.arange(len(variants))
        w = 0.26
        for i, g in enumerate(groups):
            vals = []
            for v in variants:
                r = only[(only["variant"] == v) & (only["group"] == g)]
                vals.append(float(r[metric].iloc[0]) if len(r) else np.nan)
            ax.bar(x + (i - 1) * w, vals, w, label=g.replace("_", " "), color=PALETTE.get(g, None))

        # baseline = all features (dashed line per variant)
        base_vals = []
        for v in variants:
            r = df[(df["variant"] == v) & (df["mode"] == "all_features")]
            base_vals.append(float(r[metric].iloc[0]) if len(r) else np.nan)
        for xi, bv in zip(x, base_vals):
            ax.hlines(bv, xi - 1.5 * w, xi + 1.5 * w, linestyles="--", colors="black", linewidth=1.1)

        ax.set_xticks(x)
        ax.set_xticklabels(variants)
        ax.set_ylabel(title)
        ax.set_title(f"Only-this-group performance ({title})")
        ax.grid(axis="y", linestyle=":", alpha=0.5)
        ax.legend(loc="upper right", fontsize=9, title="Feature group")

    fig.suptitle("Feature-group ablation (LightGBM). Dashed = all-features baseline.", y=1.02, fontsize=12)
    _save(fig, "fig_feature_ablation")


# ---------------------------------------------------------------------------
# 6. SHAP summary per variant
# ---------------------------------------------------------------------------
def fig_shap_summary(variant: str, shap_values: np.ndarray, X_sample: pd.DataFrame, top_n: int = 15) -> None:
    import shap

    fig = plt.figure(figsize=(8, 0.35 * top_n + 1.5))
    shap.summary_plot(
        shap_values,
        X_sample,
        max_display=top_n,
        show=False,
        plot_size=None,
    )
    plt.title(f"SHAP feature importance - variant: {variant} (LightGBM, top {top_n})", fontsize=11)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / f"fig_shap_{variant}.png", dpi=300, bbox_inches="tight")
    plt.savefig(FIGURES_DIR / f"fig_shap_{variant}.pdf", bbox_inches="tight")
    plt.close(fig)


# ---------------------------------------------------------------------------
# 7. LOPO per-project F1 distribution
# ---------------------------------------------------------------------------
def fig_lopo_per_project() -> None:
    df = pd.read_csv(TABLES_DIR / "lopo_folds.csv")
    order = ["consequence", "severity", "szz"]
    fig, ax = plt.subplots(figsize=(10, 4.5))
    data = [df.loc[df["variant"] == v, "f1"].values for v in order]
    parts = ax.boxplot(
        data,
        tick_labels=order,
        showmeans=True,
        patch_artist=True,
        medianprops={"color": "black"},
    )
    for patch, v in zip(parts["boxes"], order):
        patch.set_facecolor(PALETTE.get(v, "#ccc"))
        patch.set_alpha(0.7)

    # overlay individual projects
    for i, v in enumerate(order, start=1):
        sub = df[df["variant"] == v]
        ax.scatter(
            np.random.normal(i, 0.06, size=len(sub)),
            sub["f1"],
            alpha=0.5,
            s=22,
            color="black",
        )

    ax.set_ylabel("F1 per held-out project")
    ax.set_title("LOPO F1 distribution per variant (best model per variant, across 22 projects)")
    ax.grid(axis="y", linestyle=":", alpha=0.5)
    _save(fig, "fig_lopo_per_project")


**`src/reporting/render.py`** - Render docs/06_results.md and 07_discussion.md from artefacts.

In [ ]:
%%writefile /content/src/reporting/render.py
"""
Render the thesis-ready results chapter (``docs/06_results.md``) from
the tables and figures produced by the pipeline.

All content is *derived* from the CSV / parquet artefacts - no numbers
are typed by hand. Re-running the pipeline and then calling
``render_results()`` always yields an internally-consistent report.
"""
from __future__ import annotations

import sys
from pathlib import Path
from textwrap import dedent

import pandas as pd

sys.path.append(str(Path(__file__).resolve().parents[2]))
from config import DOCS_DIR, PROCESSED_DATA_DIR, TABLES_DIR  # noqa: E402


METRIC_HEADS = ["precision", "recall", "f1", "roc_auc", "pr_auc", "mcc", "ce_at_20"]


def _tbl_md(df: pd.DataFrame, max_rows: int | None = None, floatfmt: str = ".3f") -> str:
    if max_rows is not None:
        df = df.head(max_rows)
    return df.to_markdown(index=False, floatfmt=floatfmt)


def _best_row(df: pd.DataFrame, variant: str, by: str) -> pd.Series:
    sub = df[df["variant"] == variant].sort_values(by, ascending=False)
    return sub.iloc[0]


def render_results() -> Path:
    # ---- load all artefacts ----
    snaps = pd.read_parquet(PROCESSED_DATA_DIR / "project_snapshots.parquet")
    eligible = snaps[snaps["eligible"]]

    label_summary = pd.read_csv(TABLES_DIR / "label_summary.csv")
    label_agreement = pd.read_csv(TABLES_DIR / "label_agreement.csv")
    dataset_summary = pd.read_csv(TABLES_DIR / "dataset_summary.csv")
    within_summary = pd.read_csv(TABLES_DIR / "within_project_summary.csv")
    lopo_summary = pd.read_csv(TABLES_DIR / "lopo_summary.csv")
    lopo_gap = pd.read_csv(TABLES_DIR / "lopo_vs_within.csv")
    sensitivity = pd.read_csv(TABLES_DIR / "sensitivity_consequence.csv")
    ablation = pd.read_csv(TABLES_DIR / "feature_ablation.csv")

    within_best = {
        v: _best_row(within_summary, v, "f1_mean") for v in ("consequence", "severity", "szz")
    }
    lopo_best = {
        v: _best_row(lopo_summary, v, "f1_mean") for v in ("consequence", "severity", "szz")
    }

    # ---- assemble markdown ----
    lines: list[str] = []
    lines.append("# Chapter 6 - Results\n")
    lines.append(
        "This chapter reports the empirical results of the High-Risk Technical "
        "Debt prediction pipeline. All numbers, figures and tables are generated "
        "directly from the CSV and parquet artefacts produced by the pipeline "
        "(see ``results/tables`` and ``results/figures``) and are fully "
        "reproducible by re-running ``scripts/01_inspect_db.py`` through "
        "``scripts/10_report.py``.\n"
    )

    # ----- 6.1 Corpus and snapshot -----
    lines.append("## 6.1 Corpus and temporal snapshot\n")
    lines.append(
        f"The study covers **{len(eligible)} eligible Apache Java projects** from "
        "the Technical Debt Dataset v2.0. For every project we compute a "
        "per-project snapshot ``t`` equal to the median commit date; "
        "features are restricted to events on or before ``t`` and labels "
        "are derived from an observation window of "
        "``OBSERVATION_WINDOW_MONTHS`` (primary: 6 months).\n"
    )
    lines.append("Project snapshot details (all 22 eligible projects):\n")
    col_map = {
        "project_id": "project",
        "snapshot_date": "snapshot_t",
        "pre_snapshot_commits": "pre_commits",
        "post_snapshot_commits": "post_commits",
        "distinct_files_pre": "files_pre",
        "distinct_authors_pre": "authors_pre",
    }
    snap_tbl = eligible[list(col_map.keys())].copy()
    snap_tbl["snapshot_date"] = pd.to_datetime(snap_tbl["snapshot_date"]).dt.strftime("%Y-%m-%d")
    snap_tbl.columns = [col_map[c] for c in snap_tbl.columns]
    lines.append(_tbl_md(snap_tbl, floatfmt=".0f") + "\n")

    # ----- 6.2 Labeling -----
    lines.append("## 6.2 Three-variant labeling\n")
    lines.append(
        "Each (project, basename) pair is labeled three ways: a "
        "**consequence** label (top 20% by weighted risk score over bug-fix "
        "commits, future churn, SZZ events in the 6-month window), a "
        "**severity** label (any open SonarQube BLOCKER or CRITICAL issue at "
        "``t``), and an **SZZ** label (touched by a fault-fixing commit inside "
        "the window).\n"
    )
    lines.append("### 6.2.1 Per-project positive rates\n")
    lines.append(_tbl_md(label_summary, floatfmt=".2f") + "\n")
    lines.append("See Figure ``fig_per_project_positive_rates``.\n")

    lines.append("### 6.2.2 Label agreement\n")
    lines.append(
        "Agreement between the three variants is low (Cohen's kappa in "
        "[0.05, 0.21]), confirming that they identify largely disjoint "
        "file sets:\n"
    )
    lines.append(_tbl_md(label_agreement, floatfmt=".4f") + "\n")
    lines.append("See Figure ``fig_label_agreement_venn``.\n")

    # ----- 6.3 Datasets -----
    lines.append("## 6.3 Dataset-build leakage audit\n")
    lines.append(
        "After merging features with labels, each variant's dataset is "
        "checked for label-leakage. ``SEVERITY_LEAKY_FEATURES`` (the six "
        "per-severity counts plus ``max_severity_rank``) are dropped from the "
        "severity dataset; no leaky features remain in any variant.\n"
    )
    lines.append(_tbl_md(dataset_summary, floatfmt=".2f") + "\n")

    # ----- 6.4 Within-project -----
    lines.append("## 6.4 Within-project 10-fold cross-validation\n")
    lines.append(
        "Stratified 10-fold cross-validation on the combined "
        "(22-project, 23,911-row) dataset. Mean metrics across folds:\n"
    )
    mean_cols = [f"{m}_mean" for m in METRIC_HEADS]
    keep = ["variant", "model"] + mean_cols
    lines.append(_tbl_md(within_summary[keep], floatfmt=".3f") + "\n")
    lines.append(
        "### Within-project highlights\n"
        f"- **Consequence**: best model = **{within_best['consequence']['model']}**, "
        f"F1={within_best['consequence']['f1_mean']:.3f}, "
        f"CE@20={within_best['consequence']['ce_at_20_mean']:.3f}.\n"
        f"- **Severity**: best model = **{within_best['severity']['model']}**, "
        f"F1={within_best['severity']['f1_mean']:.3f}, "
        f"CE@20={within_best['severity']['ce_at_20_mean']:.3f}.\n"
        f"- **SZZ**: best model = **{within_best['szz']['model']}**, "
        f"F1={within_best['szz']['f1_mean']:.3f}, "
        f"CE@20={within_best['szz']['ce_at_20_mean']:.3f}.\n"
    )

    # ----- 6.5 LOPO -----
    lines.append("## 6.5 Leave-One-Project-Out cross-project validation\n")
    lines.append(
        "For each variant and model we train on 21 projects and test on the "
        "held-out project, repeating for every project. The SZZ variant covers "
        "16/22 projects because 6 projects have zero SZZ positives in their "
        "observation window.\n"
    )
    keep2 = ["variant", "model", "n_projects"] + mean_cols
    lines.append(_tbl_md(lopo_summary[keep2], floatfmt=".3f") + "\n")

    lines.append("### 6.5.1 Generalization gap\n")
    lines.append(
        "Difference between within-project and LOPO mean performance:\n"
    )
    gap_cols = ["variant", "model"] + [
        c for m in METRIC_HEADS for c in (f"{m}_within", f"{m}_lopo", f"{m}_gap")
    ]
    gap_cols = [c for c in gap_cols if c in lopo_gap.columns]
    lines.append(_tbl_md(lopo_gap[gap_cols], floatfmt=".3f") + "\n")
    lines.append(
        f"- **Consequence**: best LOPO model = **{lopo_best['consequence']['model']}**, "
        f"F1={lopo_best['consequence']['f1_mean']:.3f}, "
        f"CE@20={lopo_best['consequence']['ce_at_20_mean']:.3f}.\n"
        f"- **Severity** (best LOPO): **{lopo_best['severity']['model']}**, "
        f"F1={lopo_best['severity']['f1_mean']:.3f}, "
        f"CE@20={lopo_best['severity']['ce_at_20_mean']:.3f}.\n"
        f"- **SZZ** (best LOPO): **{lopo_best['szz']['model']}**, "
        f"F1={lopo_best['szz']['f1_mean']:.3f}, "
        f"CE@20={lopo_best['szz']['ce_at_20_mean']:.3f}.\n"
    )
    lines.append("See Figures ``fig_within_vs_lopo`` and ``fig_lopo_per_project``.\n")

    # ----- 6.6 Sensitivity -----
    lines.append("## 6.6 Sensitivity to labeling parameters\n")
    lines.append(
        "The consequence-variant default is (window=6 months, "
        "percentile=top 20%). The grid below shows LightGBM 10-fold CV "
        "performance across a 3x3 parameter sweep:\n"
    )
    keep3 = [
        "window_months",
        "percentile",
        "positive_rate_pct",
        "f1_mean",
        "roc_auc_mean",
        "pr_auc_mean",
        "ce_at_20_mean",
    ]
    keep3 = [c for c in keep3 if c in sensitivity.columns]
    lines.append(_tbl_md(sensitivity[keep3], floatfmt=".3f") + "\n")
    lines.append("See Figure ``fig_sensitivity_heatmap``.\n")

    # ----- 6.7 Ablation -----
    lines.append("## 6.7 Feature-group ablation\n")
    lines.append(
        "Three feature groups are defined: ``static_sonar`` (per-basename "
        "SonarQube aggregates at ``t``), ``historical`` (pre-``t`` Git commit "
        "process metrics), and ``project_context`` (project-level SonarQube "
        "measures at the most recent analysis <= ``t``).\n"
    )
    keep4 = ["variant", "group", "mode", "n_features"] + [f"{m}_mean" for m in METRIC_HEADS]
    lines.append(_tbl_md(ablation[keep4], floatfmt=".3f") + "\n")
    lines.append("See Figure ``fig_feature_ablation``.\n")

    # ----- 6.8 Feature importance -----
    lines.append("## 6.8 Feature importance (SHAP + permutation)\n")
    lines.append(
        "For each variant we train a single LightGBM classifier on an 80/20 "
        "stratified split and compute TreeSHAP values on the test set. The "
        "permutation importance is a secondary check (ROC-AUC drop when the "
        "column is shuffled). Tables per variant: ``shap_top15_{variant}.csv`` "
        "and ``perm_top15_{variant}.csv`` in ``results/tables``. Figures: "
        "``fig_shap_{variant}``.\n"
    )
    for i, v in enumerate(("consequence", "severity", "szz"), start=1):
        path = TABLES_DIR / f"shap_top15_{v}.csv"
        if not path.exists():
            continue
        df = pd.read_csv(path)
        lines.append(f"### 6.8.{i} Top-15 SHAP features - {v}\n")
        lines.append(_tbl_md(df, floatfmt=".4f") + "\n")

    # ----- done -----
    out_path = DOCS_DIR / "06_results.md"
    out_path.write_text("\n".join(lines), encoding="utf-8")
    return out_path


def render_discussion_scaffold() -> Path:
    """Write a skeleton of Chapter 7 that summarises key findings.

    This is a scaffold - the writing itself remains the thesis author's
    responsibility, but the framing and anchor numbers are pulled from
    the generated tables so the draft is factually grounded.
    """
    within = pd.read_csv(TABLES_DIR / "within_project_summary.csv")
    lopo = pd.read_csv(TABLES_DIR / "lopo_summary.csv")
    ablation = pd.read_csv(TABLES_DIR / "feature_ablation.csv")
    agreement = pd.read_csv(TABLES_DIR / "label_agreement.csv")

    best_cons_within = _best_row(within, "consequence", "f1_mean")
    best_cons_lopo = _best_row(lopo, "consequence", "f1_mean")

    body = dedent(
        f"""
        # Chapter 7 - Discussion (scaffold)

        ## 7.1 RQ1: Do the three label variants identify different files?

        **Answer: Yes, and the disagreement is large.** Pairwise Cohen's
        kappa between the three variants ranges from 0.05 (severity vs
        SZZ) to 0.21 (consequence vs severity). Jaccard similarity is
        at most 0.18. This is the first empirical demonstration on the
        Technical Debt Dataset v2.0 that the choice of operational
        definition for "high-risk TD" fundamentally changes which files
        are prioritised. See Table 6.5 and Figure
        ``fig_label_agreement_venn``.

        ## 7.2 RQ2: How accurately can each variant be predicted?

        **Within-project** (stratified 10-fold CV, best model):

        - Consequence: F1 = {best_cons_within['f1_mean']:.3f},
          CE@20 = {best_cons_within['ce_at_20_mean']:.3f}
          ({best_cons_within['model']})
        - Severity is near-ceiling (F1 > 0.70) and SZZ is hardest (F1
          around 0.27). The order is consistent with the intrinsic
          difficulty of each label.

        **Cross-project** (LOPO on 22 projects):

        - Consequence: F1 = {best_cons_lopo['f1_mean']:.3f},
          CE@20 = {best_cons_lopo['ce_at_20_mean']:.3f}
          ({best_cons_lopo['model']}). A top-20-percent inspection
          budget in an unseen project captures roughly 50 percent of
          files that will cause real maintenance burden in the next 6
          months - practically usable.

        ## 7.3 RQ3: Which feature families drive each variant?

        From the ablation (Table 6.7):

        - **Severity** is predicted best from *static SonarQube* features
          alone - confirming the "tautology" interpretation of severity
          labels.
        - **Consequence** and **SZZ** are predicted best from
          *historical (process)* features - matching Kamei et al. (2013)
          for JIT defect prediction.
        - Project-level context alone is weak on every variant (F1 < 0.32).

        ## 7.4 Implications

        1. Research on TD prediction that uses severity labels primarily
           benchmarks **SonarQube consistency**, not future impact.
        2. If the goal is prioritising maintenance effort, the
           **consequence framing** is a defensible alternative that
           captures different information (kappa < 0.25 against both
           baselines).
        3. A 22-project Apache corpus with 6-month windows is sufficient
           for cross-project generalisation (Herbold 2018 protocol).

        ## 7.5 Threats to validity

        - **Construct**: basename aggregation (documented in Research
          Log 2026-04-23); median basename-collision rate 37 percent.
        - **Internal**: 20-percent percentile threshold for consequence
          positives may be sensitive to project-level positive-rate
          drift - partially addressed in Section 6.6 sensitivity grid.
        - **External**: all projects are Apache Java - findings may not
          transfer to proprietary or non-Java codebases.
        - **Conclusion**: stratified K-fold allows same-project
          contamination, inflating within-project numbers; the LOPO
          numbers in Section 6.5 should be taken as the realistic
          deployment estimate.
        """
    ).strip() + "\n"

    out_path = DOCS_DIR / "07_discussion.md"
    out_path.write_text(body, encoding="utf-8")
    return out_path


In [ ]:
# Final modules cell - import everything so stage drivers can use it.
import importlib

# Force a fresh import in case the cell is re-run after editing a writefile.
for name in list(sys.modules):
    if name == 'config' or name.startswith('src.'):
        del sys.modules[name]

import config  # noqa: F401
from src.data import load_data, snapshot, clean, szz, labeling  # noqa: F401
from src.features import static_features, historical_features  # noqa: F401
from src.models import train, cross_project  # noqa: F401
from src.analysis import sensitivity, ablation, importance  # noqa: F401
from src.reporting import figures, render  # noqa: F401

print('All src/ modules imported.')
print('  config.PROCESSED_DATA_DIR =', config.PROCESSED_DATA_DIR)
print('  config.TD_DATASET_PATH    =', config.TD_DATASET_PATH)


<a id="stage-1"></a>
## Stage 1 - Database inspection

Open `td_V2.db`, list every table, record per-column metadata via `PRAGMA table_info`, and persist a 3-row sample per table. This is the canonical schema record referenced by every subsequent stage. No data transformation happens here.

Outputs:
- `results/tables/db_schema.csv` - one row per (table, column).
- `results/tables/db_table_counts.csv` - per-table row counts.
- `results/tables/db_samples/<table>.csv` - first 3 rows per table.


In [ ]:
import time
import pandas as pd
from src.data.load_data import (
    get_connection, get_row_count, get_sample_rows,
    get_table_names, get_table_schema,
)
from config import TABLES_DIR, TD_DATASET_PATH

_t0 = time.time()
_samples_dir = TABLES_DIR / 'db_samples'
_samples_dir.mkdir(parents=True, exist_ok=True)

with get_connection() as conn:
    _tables = get_table_names(conn)
    print(f'[Stage 1] Tables found: {len(_tables)}')
    _schema_rows, _count_rows = [], []
    for _t in _tables:
        _count = get_row_count(conn, _t)
        _count_rows.append({'table': _t, 'row_count': _count})
        _schema = get_table_schema(conn, _t)
        _sample = get_sample_rows(conn, _t, 3)
        _sample.to_csv(_samples_dir / f'{_t}.csv', index=False)
        _first = _sample.iloc[0] if len(_sample) else pd.Series(dtype=object)
        for _, _col in _schema.iterrows():
            _example = _first.get(_col['name'], None) if len(_first) else None
            if isinstance(_example, str) and len(_example) > 120:
                _example = _example[:117] + '...'
            _schema_rows.append({
                'table': _t, 'column': _col['name'], 'type': _col['type'],
                'notnull': bool(_col['notnull']), 'pk': bool(_col['pk']),
                'example_value': _example,
            })

_schema_df = pd.DataFrame(_schema_rows)
_counts_df = pd.DataFrame(_count_rows).sort_values('row_count', ascending=False)
_schema_df.to_csv(TABLES_DIR / 'db_schema.csv', index=False)
_counts_df.to_csv(TABLES_DIR / 'db_table_counts.csv', index=False)

RUNTIMES['stage_01_inspect'] = round(time.time() - _t0, 2)
print(f'[Stage 1] Elapsed: {RUNTIMES["stage_01_inspect"]} s')


In [ ]:
# Inspect Stage 1 outputs.
from IPython.display import display, Markdown

_counts = pd.read_csv(TABLES_DIR / 'db_table_counts.csv')
display(Markdown('**Row counts per table:**'))
display(_counts)

_schema = pd.read_csv(TABLES_DIR / 'db_schema.csv')
display(Markdown(f'**Schema: {_schema["table"].nunique()} tables, '
                 f'{len(_schema)} columns total.** First 12 columns shown:'))
display(_schema.head(12))


<a id="stage-2"></a>
## Stage 2 - Per-project snapshot selection

For every project, the master-branch commit history is loaded and a snapshot date `t` is selected as the median commit date. Pre-snapshot history feeds features; the post-snapshot 6-month window feeds labels. Projects without enough history on either side are flagged ineligible.

Eligibility thresholds: `MIN_PRE_SNAPSHOT_COMMITS = 500`, `MIN_POST_SNAPSHOT_COMMITS = 50`.

Outputs:
- `data/processed/project_snapshots.parquet` - full per-project metadata.
- `results/tables/project_stats.csv` - same data as CSV.


In [ ]:
from src.data.load_data import get_connection
from src.data.snapshot import compute_all_snapshots
from config import PROCESSED_DATA_DIR, TABLES_DIR, OBSERVATION_WINDOW_MONTHS, SNAPSHOT_STRATEGY

_snap_path = PROCESSED_DATA_DIR / 'project_snapshots.parquet'
if _snap_path.exists() and not RECOMPUTE:
    print(f'[Stage 2] Cached: {_snap_path} (set RECOMPUTE=True to regenerate)')
    snapshots_df = pd.read_parquet(_snap_path)
    RUNTIMES['stage_02_snapshot'] = 0.0
else:
    _t0 = time.time()
    print(f'[Stage 2] Strategy={SNAPSHOT_STRATEGY}, window={OBSERVATION_WINDOW_MONTHS} mo')
    with get_connection() as conn:
        snapshots_df = compute_all_snapshots(conn)
    snapshots_df.to_parquet(_snap_path, index=False)
    snapshots_df.to_csv(TABLES_DIR / 'project_stats.csv', index=False)
    RUNTIMES['stage_02_snapshot'] = round(time.time() - _t0, 2)
    print(f'[Stage 2] Elapsed: {RUNTIMES["stage_02_snapshot"]} s')


In [ ]:
# Inspect snapshot eligibility.
_eligible = snapshots_df[snapshots_df['eligible']].copy()
_excluded = snapshots_df[~snapshots_df['eligible']].copy()
display(Markdown(
    f'**Eligibility:** {len(_eligible)} of {len(snapshots_df)} projects.  '
    f'Excluded: {len(_excluded)} (reasons: '
    f"{sorted(_excluded['exclusion_reason'].dropna().unique().tolist())})."
))
_disp_cols = [
    'project_id', 'first_commit', 'last_commit', 'snapshot_date',
    'total_commits', 'pre_snapshot_commits', 'post_snapshot_commits',
    'distinct_files_pre', 'distinct_authors_pre',
]
display(_eligible[_disp_cols].sort_values('total_commits', ascending=False).reset_index(drop=True))


<a id="stage-3"></a>
## Stage 3 - Clean and basename-normalise

Convert the raw SQLite tables into tidy parquet files restricted to eligible projects and Java source files (excluding tests, generated code and build artefacts). Path normalisation extracts the **basename** from `SONAR_ISSUES.COMPONENT` and `GIT_COMMITS_CHANGES.FILE` so the two tables can join on `(project_id, basename)` (the canonical unit of analysis - documented in the thesis Threats to Construct Validity).

Outputs:
- 6 cleaned parquets in `data/processed/clean_*.parquet`.
- `results/tables/path_overlap_report.csv` - per-project basename coverage and collision-rate sanity check.


In [ ]:
from src.data.clean import (
    clean_git_commits, clean_git_commits_changes, clean_jira_issues,
    clean_sonar_issues, clean_sonar_measures_with_dates, clean_szz_with_dates,
)
from src.data.snapshot import load_snapshots

_required = [
    'clean_git_commits.parquet', 'clean_git_commits_changes.parquet',
    'clean_sonar_issues.parquet', 'clean_sonar_measures.parquet',
    'clean_szz.parquet', 'clean_jira_issues.parquet',
]
_all_present = all((PROCESSED_DATA_DIR / p).exists() for p in _required)
if _all_present and not RECOMPUTE:
    print('[Stage 3] Cached cleaned parquets present; skipping.')
    RUNTIMES['stage_03_clean'] = 0.0
else:
    _t0 = time.time()
    _snaps = load_snapshots(PROCESSED_DATA_DIR / 'project_snapshots.parquet')
    _projects = _snaps[_snaps['eligible']]['project_id'].tolist()
    print(f'[Stage 3] Cleaning {len(_projects)} eligible projects ...')
    with get_connection() as conn:
        gc = clean_git_commits(conn, _projects)
        print(f'  GIT_COMMITS:           {len(gc):>12,}')
        gcc = clean_git_commits_changes(conn, _projects, java_only=True)
        print(f'  GIT_COMMITS_CHANGES:   {len(gcc):>12,} (Java only)')
        si = clean_sonar_issues(conn, _projects, java_only=True)
        print(f'  SONAR_ISSUES:          {len(si):>12,} (Java only)')
        sm = clean_sonar_measures_with_dates(conn, _projects)
        print(f'  SONAR_MEASURES (dates):{len(sm):>12,}')
        szz_df = clean_szz_with_dates(conn, _projects, gc)
        print(f'  SZZ (dates):           {len(szz_df):>12,}')
        ji = clean_jira_issues(conn, _projects)
        print(f'  JIRA_ISSUES:           {len(ji):>12,}')

    gc.to_parquet(PROCESSED_DATA_DIR / 'clean_git_commits.parquet', index=False)
    gcc.to_parquet(PROCESSED_DATA_DIR / 'clean_git_commits_changes.parquet', index=False)
    si.to_parquet(PROCESSED_DATA_DIR / 'clean_sonar_issues.parquet', index=False)
    sm.to_parquet(PROCESSED_DATA_DIR / 'clean_sonar_measures.parquet', index=False)
    szz_df.to_parquet(PROCESSED_DATA_DIR / 'clean_szz.parquet', index=False)
    ji.to_parquet(PROCESSED_DATA_DIR / 'clean_jira_issues.parquet', index=False)

    # Basename-overlap sanity check
    _gcc_paths = gcc.groupby('PROJECT_ID')['basename'].agg(set).rename('git_basenames')
    _si_paths = si.groupby('PROJECT_ID')['basename'].agg(set).rename('sonar_basenames')
    _overlap = pd.concat([_gcc_paths, _si_paths], axis=1)
    for _c in ('git_basenames', 'sonar_basenames'):
        _overlap[_c] = _overlap[_c].apply(lambda x: x if isinstance(x, set) else set())
    _overlap['n_sonar_basenames'] = _overlap['sonar_basenames'].map(len)
    _overlap['n_git_basenames'] = _overlap['git_basenames'].map(len)
    _overlap['n_intersection'] = _overlap.apply(
        lambda r: len(r['sonar_basenames'] & r['git_basenames']), axis=1)
    _overlap['sonar_coverage_pct'] = (
        _overlap['n_intersection'] / _overlap['n_sonar_basenames'].where(_overlap['n_sonar_basenames'] > 0)
    ).round(4) * 100
    _overlap = _overlap[['n_sonar_basenames', 'n_git_basenames', 'n_intersection', 'sonar_coverage_pct']]
    _coll_rows = []
    for _pid, _sub in si.groupby('PROJECT_ID'):
        _files_per_base = _sub.groupby('basename')['file_path'].nunique()
        _coll_rows.append({
            'PROJECT_ID': _pid,
            'sonar_full_paths': int(_sub['file_path'].nunique()),
            'sonar_basenames': int(_files_per_base.shape[0]),
            'max_files_per_basename': int(_files_per_base.max()) if len(_files_per_base) else 0,
            'basename_collision_pct': (
                round((1 - _files_per_base.shape[0] / _sub['file_path'].nunique()) * 100, 1)
                if _sub['file_path'].nunique() else 0.0
            ),
        })
    _collisions = pd.DataFrame(_coll_rows).set_index('PROJECT_ID')
    _overlap = _overlap.join(_collisions)
    _overlap.reset_index().to_csv(TABLES_DIR / 'path_overlap_report.csv', index=False)

    RUNTIMES['stage_03_clean'] = round(time.time() - _t0, 2)
    print(f'[Stage 3] Elapsed: {RUNTIMES["stage_03_clean"]} s')


In [ ]:
# Inspect Stage 3 cleaned tables and basename overlap.
_overlap = pd.read_csv(TABLES_DIR / 'path_overlap_report.csv')
_min = _overlap['sonar_coverage_pct'].min()
_avg = _overlap['sonar_coverage_pct'].mean()
_med_coll = _overlap['basename_collision_pct'].median()
_max_coll = _overlap['basename_collision_pct'].max()
display(Markdown(
    '**Sonar -> Git basename coverage:** '
    f'min={_min:.1f}% / avg={_avg:.1f}%.  '
    f'**Basename collision (multiple full-paths -> same basename):** '
    f'median={_med_coll:.1f}% / max={_max_coll:.1f}%.'
))
display(_overlap.sort_values('sonar_coverage_pct', ascending=False).reset_index(drop=True))

_clean_summary = pd.DataFrame([
    {'parquet': p, 'rows': len(pd.read_parquet(PROCESSED_DATA_DIR / p))}
    for p in [
        'clean_git_commits.parquet', 'clean_git_commits_changes.parquet',
        'clean_sonar_issues.parquet', 'clean_sonar_measures.parquet',
        'clean_szz.parquet', 'clean_jira_issues.parquet',
    ]
])
display(Markdown('**Cleaned parquet row counts:**'))
display(_clean_summary)


<a id="stage-4"></a>
## Stage 4 - Three-variant labelling

Materialise three label variants per (project, basename):

- **Consequence (primary)** - top 20% of basenames within each project by a weighted risk score combining future bug-fix commits (0.5), future churn (0.3) and SZZ events (0.2) within the 6-month observation window.
- **Severity** - any open SonarQube `BLOCKER` or `CRITICAL` issue at `t`.
- **SZZ** - touched by at least one fault-fixing commit in the 6-month window.

Pairwise Cohen's kappa and Jaccard similarity are computed to quantify how disjoint the three variants are.

Outputs: `data/processed/labels_{consequence,severity,szz}.parquet`, `results/tables/label_summary.csv`, `results/tables/label_agreement.csv`.


In [ ]:
from src.data.labeling import (
    compute_consequence_labels, compute_severity_labels, compute_szz_labels, label_agreement,
)
from config import OBSERVATION_WINDOW_MONTHS, HIGH_RISK_PERCENTILE, SEVERITY_BASELINE_LEVELS
from tqdm.auto import tqdm

def _load_clean_for_labeling():
    _commits = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_git_commits.parquet')
    _changes = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_git_commits_changes.parquet')
    _sonar_issues = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_sonar_issues.parquet')
    _szz = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_szz.parquet')
    _jira = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_jira_issues.parquet')
    for _df, _cols in (
        (_commits, ['AUTHOR_DATE', 'COMMITTER_DATE']),
        (_changes, ['DATE']),
        (_sonar_issues, ['CREATION_DATE', 'CLOSE_DATE']),
        (_szz, ['fix_date', 'induce_date']),
        (_jira, ['CREATION_DATE', 'RESOLUTION_DATE', 'UPDATE_DATE', 'COMMIT_DATE']),
    ):
        for _c in _cols:
            if _c in _df.columns and _df[_c].dtype.kind == 'M' and _df[_c].dt.tz is None:
                _df[_c] = _df[_c].dt.tz_localize('UTC')
    return {'commits': _commits, 'changes': _changes,
            'sonar_issues': _sonar_issues, 'szz': _szz, 'jira': _jira}

_required = ['labels_consequence.parquet', 'labels_severity.parquet', 'labels_szz.parquet']
_have_all = all((PROCESSED_DATA_DIR / p).exists() for p in _required)
if _have_all and not RECOMPUTE:
    print('[Stage 4] Cached label parquets present; skipping.')
    RUNTIMES['stage_04_label'] = 0.0
else:
    _t0 = time.time()
    print(f'[Stage 4] window={OBSERVATION_WINDOW_MONTHS} mo, percentile=top {HIGH_RISK_PERCENTILE}%, '
          f'severity={SEVERITY_BASELINE_LEVELS}')
    _data = _load_clean_for_labeling()
    _snaps = load_snapshots(PROCESSED_DATA_DIR / 'project_snapshots.parquet')
    _eligible = _snaps[_snaps['eligible']].sort_values('project_id')
    _cons_parts, _sev_parts, _szz_parts, _summary = [], [], [], []
    for _, _row in tqdm(list(_eligible.iterrows()), desc='Labelling projects'):
        _pid, _t = _row['project_id'], _row['snapshot_date']
        _cons = compute_consequence_labels(_pid, _t, _data['commits'], _data['changes'],
                                           _data['szz'], _data['jira'])
        _sev = compute_severity_labels(_pid, _t, _data['sonar_issues'], _data['changes'])
        _szzl = compute_szz_labels(_pid, _t, _data['changes'], _data['szz'])
        _cons_parts.append(_cons); _sev_parts.append(_sev); _szz_parts.append(_szzl)
        _summary.append({
            'project_id': _pid, 'n_basenames': len(_cons),
            'consequence_positives': int(_cons['is_high_risk'].sum()),
            'consequence_rate_pct': round(100 * _cons['is_high_risk'].mean(), 2),
            'severity_positives': int(_sev['is_high_risk'].sum()),
            'severity_rate_pct': round(100 * _sev['is_high_risk'].mean(), 2),
            'szz_positives': int(_szzl['is_high_risk'].sum()),
            'szz_rate_pct': round(100 * _szzl['is_high_risk'].mean(), 2),
        })
    _cons_all = pd.concat(_cons_parts, ignore_index=True)
    _sev_all = pd.concat(_sev_parts, ignore_index=True)
    _szz_all = pd.concat(_szz_parts, ignore_index=True)
    _cons_all.to_parquet(PROCESSED_DATA_DIR / 'labels_consequence.parquet', index=False)
    _sev_all.to_parquet(PROCESSED_DATA_DIR / 'labels_severity.parquet', index=False)
    _szz_all.to_parquet(PROCESSED_DATA_DIR / 'labels_szz.parquet', index=False)
    pd.DataFrame(_summary).to_csv(TABLES_DIR / 'label_summary.csv', index=False)
    _agr = label_agreement({'consequence': _cons_all, 'severity': _sev_all, 'szz': _szz_all})
    _agr.to_csv(TABLES_DIR / 'label_agreement.csv', index=False)
    RUNTIMES['stage_04_label'] = round(time.time() - _t0, 2)
    print(f'[Stage 4] Elapsed: {RUNTIMES["stage_04_label"]} s')


In [ ]:
# Inspect labelling outcomes.
_summary = pd.read_csv(TABLES_DIR / 'label_summary.csv')
_agr = pd.read_csv(TABLES_DIR / 'label_agreement.csv')
display(Markdown('**Per-project positive rates (%):**'))
display(_summary.style.format({
    'consequence_rate_pct': '{:.2f}',
    'severity_rate_pct': '{:.2f}',
    'szz_rate_pct': '{:.2f}',
}))
display(Markdown('**Pairwise label agreement (Cohen kappa, Jaccard):**'))
display(_agr)

for _v in ('consequence', 'severity', 'szz'):
    _df = pd.read_parquet(PROCESSED_DATA_DIR / f'labels_{_v}.parquet')
    print(f'  labels_{_v:<11} rows={len(_df):>6,} positive_rate={100*_df["is_high_risk"].mean():.2f}%')


<a id="stage-5"></a>
## Stage 5 - Static and historical features

Build snapshot-aware features at `(project_id, basename)` granularity. Two families are produced:

- **Static** - per-basename SonarQube issue aggregates open at `t`, plus project-level SonarQube context from the latest analysis with `analysis_date <= t`, plus a Git-derived size proxy.
- **Historical (process)** - per-basename Git commit statistics restricted to commits with `AUTHOR_DATE <= t`: total commits, contributors, churn, recency windows (30/90 d), age, ownership ratio, distributional churn statistics.

Outputs: `data/processed/features_{static,historical}.parquet`, `results/tables/feature_summary.csv`.


In [ ]:
from src.features.static_features import build_static_features_for_project
from src.features.historical_features import build_historical_features_for_project

def _load_clean_for_features():
    _commits = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_git_commits.parquet')
    _changes = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_git_commits_changes.parquet')
    _issues = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_sonar_issues.parquet')
    _measures = pd.read_parquet(PROCESSED_DATA_DIR / 'clean_sonar_measures.parquet')
    for _df, _cols in (
        (_commits, ['AUTHOR_DATE', 'COMMITTER_DATE']),
        (_changes, ['DATE']),
        (_issues, ['CREATION_DATE', 'CLOSE_DATE']),
        (_measures, ['analysis_date']),
    ):
        for _c in _cols:
            if _c in _df.columns and _df[_c].dtype.kind == 'M' and _df[_c].dt.tz is None:
                _df[_c] = _df[_c].dt.tz_localize('UTC')
    return {'commits': _commits, 'changes': _changes,
            'sonar_issues': _issues, 'sonar_measures': _measures}

_required = ['features_static.parquet', 'features_historical.parquet']
_have_all = all((PROCESSED_DATA_DIR / p).exists() for p in _required)
if _have_all and not RECOMPUTE:
    print('[Stage 5] Cached feature parquets present; skipping.')
    RUNTIMES['stage_05_features'] = 0.0
else:
    _t0 = time.time()
    _data = _load_clean_for_features()
    _snaps = load_snapshots(PROCESSED_DATA_DIR / 'project_snapshots.parquet')
    _eligible = _snaps[_snaps['eligible']].sort_values('project_id')
    _static_parts, _hist_parts, _summary = [], [], []
    for _, _row in tqdm(list(_eligible.iterrows()), desc='Building features'):
        _pid, _t = _row['project_id'], _row['snapshot_date']
        _ts = time.time()
        _static_df = build_static_features_for_project(
            _pid, _t, _data['sonar_issues'], _data['sonar_measures'], _data['changes'])
        _hist_df = build_historical_features_for_project(
            _pid, _t, _data['commits'], _data['changes'])
        _static_parts.append(_static_df); _hist_parts.append(_hist_df)
        _summary.append({
            'project_id': _pid, 'n_basenames': len(_static_df),
            'static_cols': len(_static_df.columns),
            'hist_cols': len(_hist_df.columns),
            'mean_n_issues_open': round(float(_static_df['n_issues_open'].mean()), 2)
                if 'n_issues_open' in _static_df.columns else 0.0,
            'mean_total_commits_pre': round(float(_hist_df['total_commits_pre'].mean()), 2)
                if 'total_commits_pre' in _hist_df.columns else 0.0,
            'elapsed_s': round(time.time() - _ts, 2),
        })
    _static_all = pd.concat(_static_parts, ignore_index=True)
    _hist_all = pd.concat(_hist_parts, ignore_index=True)
    _static_all.to_parquet(PROCESSED_DATA_DIR / 'features_static.parquet', index=False)
    _hist_all.to_parquet(PROCESSED_DATA_DIR / 'features_historical.parquet', index=False)
    pd.DataFrame(_summary).to_csv(TABLES_DIR / 'feature_summary.csv', index=False)
    RUNTIMES['stage_05_features'] = round(time.time() - _t0, 2)
    print(f'[Stage 5] Elapsed: {RUNTIMES["stage_05_features"]} s '
          f'(static rows={len(_static_all):,}, hist rows={len(_hist_all):,})')


In [ ]:
# Inspect feature shapes and a sample row.
_static = pd.read_parquet(PROCESSED_DATA_DIR / 'features_static.parquet')
_hist = pd.read_parquet(PROCESSED_DATA_DIR / 'features_historical.parquet')
display(Markdown(
    f'**features_static**: {_static.shape[0]:,} rows x {_static.shape[1]} cols  -  '
    f'**features_historical**: {_hist.shape[0]:,} rows x {_hist.shape[1]} cols.'
))
display(Markdown('**Feature summary per project:**'))
display(pd.read_csv(TABLES_DIR / 'feature_summary.csv'))
display(Markdown('**Sample static feature row:**'))
display(_static.head(3))
display(Markdown('**Sample historical feature row:**'))
display(_hist.head(3))


<a id="stage-6"></a>
## Stage 6 - Dataset assembly with leakage audit

Merge static and historical features with each label variant; for each variant, drop columns that would leak the label (e.g. severity counts for the severity variant). Missing project-level context is median-imputed and an indicator `has_project_context` is added.

An audit row is written per variant verifying:
- No leaky feature is retained.
- The post-merge positive rate matches Stage 4.
- The number of training features.

Outputs: `data/processed/dataset_{consequence,severity,szz}.parquet`, `results/tables/dataset_summary.csv`.


In [ ]:
from config import SEVERITY_LEAKY_FEATURES, SZZ_LEAKY_FEATURES
import numpy as np

_KEY_COLS = ['project_id', 'basename']
_LABEL_COLS = ['is_high_risk']

def _load_features():
    _static = pd.read_parquet(PROCESSED_DATA_DIR / 'features_static.parquet')
    _hist = pd.read_parquet(PROCESSED_DATA_DIR / 'features_historical.parquet')
    if 'snapshot_date' in _static.columns and 'snapshot_date' in _hist.columns:
        _hist = _hist.drop(columns=['snapshot_date'])
    return _static.merge(_hist, on=_KEY_COLS, how='outer')

def _prepare_labels(kind):
    _df = pd.read_parquet(PROCESSED_DATA_DIR / f'labels_{kind}.parquet')
    _keep = _KEY_COLS + ['is_high_risk']
    if kind == 'consequence' and 'risk_score' in _df.columns:
        _keep.append('risk_score')
    return _df[_keep]

def _drop_leaky(df, variant):
    if variant == 'severity':
        return df.drop(columns=[c for c in SEVERITY_LEAKY_FEATURES if c in df.columns])
    if variant == 'szz':
        return df.drop(columns=[c for c in SZZ_LEAKY_FEATURES if c in df.columns])
    return df

def _fill_missing_context(df):
    _proj_cols = [c for c in df.columns if c.startswith('project_') and c != 'project_id']
    if not _proj_cols:
        df['has_project_context'] = 1
        return df
    _ctx = df[_proj_cols].notna().any(axis=1)
    df = df.assign(has_project_context=_ctx.astype('int64'))
    _num = df[_proj_cols].select_dtypes(include='number').columns.tolist()
    if _num:
        df[_num] = df[_num].fillna(df[_num].median())
    _non_num = [c for c in _proj_cols if c not in _num]
    if _non_num:
        df = df.drop(columns=_non_num)
    return df

_t0 = time.time()
_features = _load_features()
_audits = []
for _variant in ('consequence', 'severity', 'szz'):
    _labels = _prepare_labels(_variant)
    _df = _features.merge(_labels, on=_KEY_COLS, how='inner')
    _df = _drop_leaky(_df, _variant)
    _df = _fill_missing_context(_df)
    _forbidden = {'severity': set(SEVERITY_LEAKY_FEATURES),
                  'szz': set(SZZ_LEAKY_FEATURES),
                  'consequence': set()}[_variant]
    _retained = sorted(set(_df.columns) & _forbidden)
    _n_features = len(_df.columns) - len(_KEY_COLS) - len(_LABEL_COLS)
    if _variant == 'consequence' and 'risk_score' in _df.columns:
        _n_features -= 1
    _audits.append({
        'variant': _variant,
        'rows': len(_df),
        'columns_total': len(_df.columns),
        'n_features': _n_features,
        'positive_rate_pct': round(100 * float(_df['is_high_risk'].mean()), 2),
        'positives': int(_df['is_high_risk'].sum()),
        'retained_leaky_cols': _retained,
        'cols_with_any_na': int((_df.isna().any()).sum()),
    })
    _df.to_parquet(PROCESSED_DATA_DIR / f'dataset_{_variant}.parquet', index=False)

_summary = pd.DataFrame(_audits)
_summary['retained_leaky_cols'] = _summary['retained_leaky_cols'].apply(
    lambda xs: ';'.join(xs) if xs else '')
_summary.to_csv(TABLES_DIR / 'dataset_summary.csv', index=False)
RUNTIMES['stage_06_dataset'] = round(time.time() - _t0, 2)
print(f'[Stage 6] Elapsed: {RUNTIMES["stage_06_dataset"]} s')


In [ ]:
# Inspect leakage audit and dataset summaries.
_summary = pd.read_csv(TABLES_DIR / 'dataset_summary.csv')
display(Markdown('**Per-variant leakage audit:**'))
display(_summary)

_assert = _summary['retained_leaky_cols'].fillna('').eq('').all()
display(Markdown(
    f'**Leakage audit verdict:** '
    f"{'PASSED - no leaky features retained.' if _assert else 'FAILED - inspect summary above.'}"
))


<a id="stage-7"></a>
## Stage 7 - Within-project 10-fold cross-validation

Stratified 10-fold CV on the combined dataset for each `(variant, model)` pair. Five model families are evaluated: logistic regression, decision tree, random forest, XGBoost and LightGBM. The metric battery is precision, recall, F1, ROC-AUC, PR-AUC, MCC, and CE@20 (Cost-Effectiveness at top-20%).

Outputs: `results/tables/within_project_folds.csv`, `results/tables/within_project_summary.csv`.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
from src.models.train import (
    fold_results_to_frame, load_variant_matrix, stratified_kfold_cv, summarize,
)
from config import CV_FOLDS, LABEL_VARIANTS

_MODELS_W = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm']
_t0 = time.time()
_all_folds = []
for _variant in LABEL_VARIANTS:
    _X, _y, _ = load_variant_matrix(_variant)
    print(f'[Stage 7] {_variant:<11} rows={len(_X):,} feats={_X.shape[1]} '
          f'positives={int(_y.sum())} ({100*_y.mean():.2f}%)')
    for _m in _MODELS_W:
        _t1 = time.time()
        _r = stratified_kfold_cv(_variant, _m, _X, _y, n_splits=CV_FOLDS)
        if not _r:
            print(f'   {_m:<20} SKIPPED (not installed)')
            continue
        _df = fold_results_to_frame(_r)
        _means = _df[['precision','recall','f1','roc_auc','pr_auc','mcc','ce_at_20']].mean()
        print(f'   {_m:<20} F1={_means["f1"]:.3f} ROC={_means["roc_auc"]:.3f} '
              f'PR={_means["pr_auc"]:.3f} MCC={_means["mcc"]:.3f} CE@20={_means["ce_at_20"]:.3f} '
              f'({time.time()-_t1:.1f}s)')
        _all_folds.append(_df)

_fold_df = pd.concat(_all_folds, ignore_index=True)
_fold_df.to_csv(TABLES_DIR / 'within_project_folds.csv', index=False)
summarize(_fold_df).to_csv(TABLES_DIR / 'within_project_summary.csv', index=False)
RUNTIMES['stage_07_within'] = round(time.time() - _t0, 2)
print(f'[Stage 7] Elapsed: {RUNTIMES["stage_07_within"]} s')


In [ ]:
# Inspect within-project summary (mean across folds, 3 d.p.).
_w = pd.read_csv(TABLES_DIR / 'within_project_summary.csv')
_mean_cols = [c for c in _w.columns if c.endswith('_mean')]
_pretty = _w[['variant', 'model'] + _mean_cols].copy()
_pretty[_mean_cols] = _pretty[_mean_cols].round(3)
display(Markdown('**Within-project 10-fold CV (mean of folds):**'))
display(_pretty)


<a id="stage-8"></a>
## Stage 8 - Leave-One-Project-Out cross-project validation

For each `(variant, model)` pair, hold out one project at a time and train on the remaining 21. The realistic deployment metric: how well does a model trained on existing projects perform on a brand-new project?

After computing LOPO means, a side-by-side `lopo_vs_within.csv` table is built (the **generalization gap**: within-project minus LOPO).

Outputs: `results/tables/lopo_folds.csv`, `lopo_summary.csv`, `lopo_vs_within.csv`.


In [ ]:
from src.models.cross_project import (
    lopo_cv, lopo_results_to_frame, lopo_summary,
)
_METRIC_COLS = ['precision','recall','f1','roc_auc','pr_auc','mcc','ce_at_20']
_MODELS_LOPO = ['logistic_regression', 'decision_tree', 'random_forest', 'xgboost', 'lightgbm']
_t0 = time.time()
_all_folds = []
for _variant in LABEL_VARIANTS:
    print(f'[Stage 8] variant={_variant}')
    for _m in _MODELS_LOPO:
        _t1 = time.time()
        _r = lopo_cv(_variant, _m)
        if not _r:
            print(f'   {_m:<20} SKIPPED'); continue
        _df = lopo_results_to_frame(_r)
        _means = _df[_METRIC_COLS].mean()
        print(f'   {_m:<20} projs={len(_df):>2} F1={_means["f1"]:.3f} '
              f'ROC={_means["roc_auc"]:.3f} PR={_means["pr_auc"]:.3f} '
              f'MCC={_means["mcc"]:.3f} CE@20={_means["ce_at_20"]:.3f} '
              f'({time.time()-_t1:.1f}s)')
        _all_folds.append(_df)

_fold_df = pd.concat(_all_folds, ignore_index=True)
_fold_df.to_csv(TABLES_DIR / 'lopo_folds.csv', index=False)
_summary = lopo_summary(_fold_df)
_summary.to_csv(TABLES_DIR / 'lopo_summary.csv', index=False)

# Generalization gap
_within_path = TABLES_DIR / 'within_project_summary.csv'
if _within_path.exists():
    _wp = pd.read_csv(_within_path)
    _keep = ['variant', 'model'] + [f'{m}_mean' for m in _METRIC_COLS]
    _wp = _wp[_keep].rename(columns={f'{m}_mean': f'{m}_within' for m in _METRIC_COLS})
    _lp = _summary[['variant', 'model'] + [f'{m}_mean' for m in _METRIC_COLS]].rename(
        columns={f'{m}_mean': f'{m}_lopo' for m in _METRIC_COLS})
    _gap = _wp.merge(_lp, on=['variant', 'model'], how='outer')
    for _m in _METRIC_COLS:
        _gap[f'{_m}_gap'] = (_gap[f'{_m}_within'] - _gap[f'{_m}_lopo']).round(3)
        _gap[f'{_m}_within'] = _gap[f'{_m}_within'].round(3)
        _gap[f'{_m}_lopo'] = _gap[f'{_m}_lopo'].round(3)
    _gap.to_csv(TABLES_DIR / 'lopo_vs_within.csv', index=False)

RUNTIMES['stage_08_lopo'] = round(time.time() - _t0, 2)
print(f'[Stage 8] Elapsed: {RUNTIMES["stage_08_lopo"]} s')


In [ ]:
# Inspect LOPO summary and generalization gap.
_lopo = pd.read_csv(TABLES_DIR / 'lopo_summary.csv')
_mean_cols = [c for c in _lopo.columns if c.endswith('_mean')]
_pretty = _lopo[['variant', 'model', 'n_projects'] + _mean_cols].copy()
_pretty[_mean_cols] = _pretty[_mean_cols].round(3)
display(Markdown('**LOPO summary (mean across held-out projects):**'))
display(_pretty)

_gap_path = TABLES_DIR / 'lopo_vs_within.csv'
if _gap_path.exists():
    _gap = pd.read_csv(_gap_path)
    _gap_cols = ['variant','model','f1_within','f1_lopo','f1_gap',
                 'ce_at_20_within','ce_at_20_lopo','ce_at_20_gap',
                 'pr_auc_within','pr_auc_lopo','pr_auc_gap']
    _gap_cols = [c for c in _gap_cols if c in _gap.columns]
    display(Markdown('**Generalization gap (within - LOPO):**'))
    display(_gap[_gap_cols])


<a id="stage-9"></a>
## Stage 9 - Sensitivity grid and feature-group ablation

Two robustness checks:

1. **Sensitivity grid** (consequence variant only): re-derive labels for every (window, percentile) in `{3,6,12} x {10,20,30}`, retrain LightGBM with stratified 10-fold CV, and tabulate the metric battery for each cell. A robust pipeline shows stable or gracefully-degrading numbers across the grid.
2. **Feature-group ablation** (all three variants): train with only one of `{static_sonar, historical, project_context}` at a time and with each group removed, against the all-features baseline. This isolates which family carries the predictive signal for each label variant.

Outputs: `results/tables/sensitivity_consequence.csv`, `results/tables/feature_ablation.csv`.


In [ ]:
from src.analysis.sensitivity import run_sensitivity_grid
from src.analysis.ablation import run_ablation

_t0 = time.time()
print('[Stage 9] Sensitivity grid (consequence, LightGBM)')
_grid = run_sensitivity_grid(windows=(3, 6, 12), percentiles=(10.0, 20.0, 30.0))
_grid.to_csv(TABLES_DIR / 'sensitivity_consequence.csv', index=False)

_ablation_parts = []
for _variant in LABEL_VARIANTS:
    _t1 = time.time()
    print(f'[Stage 9] Feature ablation: variant={_variant}')
    _df = run_ablation(_variant, model_name='lightgbm')
    _ablation_parts.append(_df)
    print(f'   ({time.time() - _t1:.1f}s)')

_ablation = pd.concat(_ablation_parts, ignore_index=True)
_ablation.to_csv(TABLES_DIR / 'feature_ablation.csv', index=False)
RUNTIMES['stage_09_sensitivity'] = round(time.time() - _t0, 2)
print(f'[Stage 9] Elapsed: {RUNTIMES["stage_09_sensitivity"]} s')


In [ ]:
# Inspect sensitivity grid (3x3) and ablation table (3 d.p.).
_grid = pd.read_csv(TABLES_DIR / 'sensitivity_consequence.csv')
_keep = ['window_months','percentile','positive_rate_pct',
         'f1_mean','roc_auc_mean','pr_auc_mean','mcc_mean','ce_at_20_mean']
_keep = [c for c in _keep if c in _grid.columns]
_grid_disp = _grid[_keep].copy()
_num_cols = [c for c in _grid_disp.columns if _grid_disp[c].dtype.kind == 'f']
_grid_disp[_num_cols] = _grid_disp[_num_cols].round(3)
display(Markdown('**Sensitivity grid (consequence variant):**'))
display(_grid_disp)

_abl = pd.read_csv(TABLES_DIR / 'feature_ablation.csv')
_keep2 = ['variant','group','mode','n_features',
          'f1_mean','roc_auc_mean','pr_auc_mean','ce_at_20_mean']
_keep2 = [c for c in _keep2 if c in _abl.columns]
_abl_disp = _abl[_keep2].copy()
_num_cols2 = [c for c in _abl_disp.columns if _abl_disp[c].dtype.kind == 'f']
_abl_disp[_num_cols2] = _abl_disp[_num_cols2].round(3)
display(Markdown('**Feature-group ablation:**'))
display(_abl_disp)


<a id="stage-10"></a>
## Stage 10 - SHAP, figures and rendered reports

Final reporting stage:

1. **SHAP + permutation importance** per variant: train one LightGBM on an 80/20 stratified split, compute TreeSHAP and permutation importance, save top-15 tables and full tables.
2. **Seven publication-ready figures** (300 DPI PNG + PDF): label-agreement Venn, per-project positive rates, within-vs-LOPO dot-plot, sensitivity heatmap, ablation bar chart, three SHAP summaries (one per variant), and LOPO per-project F1 box-plot.
3. **Auto-generated reports**: `docs/06_results.md` (every number derived from the CSV/parquet artefacts) and `docs/07_discussion.md` (scaffold).


In [ ]:
from src.analysis.importance import compute_importance
from src.reporting import figures as fig
from src.reporting.render import render_results, render_discussion_scaffold

_t0 = time.time()
for _v in ('consequence', 'severity', 'szz'):
    print(f'[Stage 10.1] SHAP / permutation importance: {_v}')
    _out = compute_importance(_v, model_name='lightgbm')
    _out['shap_summary'].head(15).to_csv(TABLES_DIR / f'shap_top15_{_v}.csv', index=False)
    _out['permutation'].head(15).to_csv(TABLES_DIR / f'perm_top15_{_v}.csv', index=False)
    _out['shap_summary'].to_csv(TABLES_DIR / f'shap_full_{_v}.csv', index=False)
    _out['permutation'].to_csv(TABLES_DIR / f'perm_full_{_v}.csv', index=False)
    fig.fig_shap_summary(_v, _out['shap_values'], _out['X_sample'], top_n=15)
    print(f'   top SHAP: {", ".join(_out["shap_summary"]["feature"].head(5).tolist())}')

print('[Stage 10.2] Global figures')
fig.fig_label_agreement_venn();        print('   - fig_label_agreement_venn')
fig.fig_per_project_positive_rates();  print('   - fig_per_project_positive_rates')
fig.fig_within_vs_lopo();              print('   - fig_within_vs_lopo')
fig.fig_sensitivity_heatmap();         print('   - fig_sensitivity_heatmap')
fig.fig_feature_ablation();            print('   - fig_feature_ablation')
fig.fig_lopo_per_project();            print('   - fig_lopo_per_project')

print('[Stage 10.3] Render docs/06_results.md + docs/07_discussion.md')
_results_path = render_results()
_disc_path = render_discussion_scaffold()
print(f'   - {_results_path}')
print(f'   - {_disc_path}')

RUNTIMES['stage_10_report'] = round(time.time() - _t0, 2)
print(f'[Stage 10] Elapsed: {RUNTIMES["stage_10_report"]} s')


In [ ]:
# Render every figure inline and preview the auto-generated results report.
from IPython.display import Image
from config import FIGURES_DIR, DOCS_DIR

_figs_in_order = [
    'fig_label_agreement_venn',
    'fig_per_project_positive_rates',
    'fig_within_vs_lopo',
    'fig_sensitivity_heatmap',
    'fig_feature_ablation',
    'fig_shap_consequence',
    'fig_shap_severity',
    'fig_shap_szz',
    'fig_lopo_per_project',
]
for _name in _figs_in_order:
    _png = FIGURES_DIR / f'{_name}.png'
    if _png.exists():
        display(Markdown(f'**{_name}**'))
        display(Image(filename=str(_png)))
    else:
        print(f'(missing) {_png}')

_results_md = (DOCS_DIR / '06_results.md').read_text(encoding='utf-8')
_preview = '\n'.join(_results_md.splitlines()[:80])
display(Markdown('---\n## Preview: docs/06_results.md (first 80 lines)\n'))
display(Markdown(_preview))


<a id="finalisation"></a>
## Finalisation - runtime, zip and copy to Drive

Persist a per-stage runtime record, build a single zip archive containing only the experimental outputs, and copy it back to Drive at `MyDrive/td_pipeline/outputs/`.


In [ ]:
# Persist per-stage runtime record.
_total_s = round(sum(RUNTIMES.values()), 2)
_runtime_record = {
    'per_stage_seconds': RUNTIMES,
    'total_seconds': _total_s,
    'total_minutes': round(_total_s / 60, 2),
    'finished_at_utc': datetime.datetime.utcnow().isoformat() + 'Z',
}
(CONTENT / 'results' / 'runtime.json').write_text(
    json.dumps(_runtime_record, indent=2), encoding='utf-8')
print(json.dumps(_runtime_record, indent=2))


In [ ]:
# Build the zip of experimental artefacts (no thesis prose included).
import zipfile, glob

_ts = datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')
_zip_name = f'td_pipeline_outputs_{_ts}.zip'
_zip_path = CONTENT / _zip_name

_INCLUDE_GLOBS = [
    'data/processed/*.parquet',
    'results/tables/*.csv',
    'results/tables/db_samples/*.csv',
    'results/figures/*.png',
    'results/figures/*.pdf',
    'results/env_receipt.json',
    'results/runtime.json',
]
_INCLUDE_DOCS = ['docs/06_results.md', 'docs/07_discussion.md']

with zipfile.ZipFile(_zip_path, 'w', zipfile.ZIP_DEFLATED) as _zf:
    _written = 0
    for _g in _INCLUDE_GLOBS:
        for _p in sorted(glob.glob(str(CONTENT / _g))):
            _arcname = Path(_p).relative_to(CONTENT).as_posix()
            _zf.write(_p, _arcname); _written += 1
    for _doc_rel in _INCLUDE_DOCS:
        _p = CONTENT / _doc_rel
        if _p.exists():
            _zf.write(_p, _doc_rel); _written += 1

print(f'Wrote {_written} files to {_zip_path}')
print(f'Zip size: {_zip_path.stat().st_size / (1024**2):.1f} MB')


In [ ]:
# Copy the zip to Drive so the user retains it after the Colab VM is recycled.
_outputs_dir = DRIVE_ROOT / 'outputs'
_outputs_dir.mkdir(parents=True, exist_ok=True)
_dest = _outputs_dir / _zip_path.name
shutil.copy2(_zip_path, _dest)
print(f'Copied to Drive: {_dest}')


## Artefact inventory

After successful execution, the zip in `MyDrive/td_pipeline/outputs/` contains exactly:

**`data/processed/`** - intermediate parquet tables
- `project_snapshots.parquet` - per-project snapshot dates and eligibility.
- `clean_git_commits.parquet`, `clean_git_commits_changes.parquet`, `clean_sonar_issues.parquet`, `clean_sonar_measures.parquet`, `clean_szz.parquet`, `clean_jira_issues.parquet` - cleaned source tables.
- `labels_consequence.parquet`, `labels_severity.parquet`, `labels_szz.parquet` - the three label variants.
- `features_static.parquet`, `features_historical.parquet` - feature matrices.
- `dataset_consequence.parquet`, `dataset_severity.parquet`, `dataset_szz.parquet` - per-variant training matrices (after leakage audit).

**`results/tables/`** - all CSV result tables
- `db_schema.csv`, `db_table_counts.csv`, `db_samples/<table>.csv` (Stage 1).
- `project_stats.csv` (Stage 2).
- `path_overlap_report.csv` (Stage 3).
- `label_summary.csv`, `label_agreement.csv` (Stage 4).
- `feature_summary.csv` (Stage 5).
- `dataset_summary.csv` (Stage 6).
- `within_project_folds.csv`, `within_project_summary.csv` (Stage 7).
- `lopo_folds.csv`, `lopo_summary.csv`, `lopo_vs_within.csv` (Stage 8).
- `sensitivity_consequence.csv`, `feature_ablation.csv` (Stage 9).
- `shap_top15_<variant>.csv`, `shap_full_<variant>.csv`, `perm_top15_<variant>.csv`, `perm_full_<variant>.csv` (Stage 10).

**`results/figures/`** - 9 figures, each as PNG + PDF (300 DPI)
- `fig_label_agreement_venn`, `fig_per_project_positive_rates`, `fig_within_vs_lopo`, `fig_sensitivity_heatmap`, `fig_feature_ablation`, `fig_shap_consequence`, `fig_shap_severity`, `fig_shap_szz`, `fig_lopo_per_project`.

**`docs/`** - exactly two auto-generated markdown files
- `06_results.md`, `07_discussion.md`.

**Top-level**
- `results/env_receipt.json` - Python / package versions, DB fingerprint, random seed.
- `results/runtime.json` - per-stage wall-clock breakdown.
